In [1]:
!pip install adversarial-robustness-toolbox -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.6 MB/s eta 0:00:00


In [2]:

"""
Single-file version for Kaggle notebook execution.
Copy this entire file to a Kaggle notebook cell.

Evaluates RL Co-Evolution IDS under Concept Drift scenarios.

Optimized for Kaggle GPU/TPU accelerator.
"""

import os
import sys
import json
import time
import copy
import logging
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional
from collections import deque

# Kaggle accelerator optimizations
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # Disable for faster async execution
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce TF logging if present

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import matplotlib.pyplot as plt

# SciPy for advanced drift metrics
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon
try:
    from scipy.linalg import sqrtm
except ImportError:
    sqrtm = None  # Fallback for older scipy versions

warnings.filterwarnings("ignore")

# =============================================================================
# KAGGLE ACCELERATOR CONFIGURATION
# =============================================================================

# Check for mixed precision support (Kaggle P100/T4/TPU)
USE_AMP = torch.cuda.is_available() and hasattr(torch.cuda, 'amp')
if USE_AMP:
    from torch.cuda.amp import autocast, GradScaler
    print("✅ Mixed Precision (AMP) enabled for faster GPU inference")

# Optimize CUDA settings for Kaggle
if torch.cuda.is_available():
    # Enable cuDNN benchmark for consistent input sizes
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # Faster, less deterministic
    # Set default tensor type to CUDA
    torch.set_default_dtype(torch.float32)
    print(f"✅ CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Batch size optimization for Kaggle GPUs
KAGGLE_BATCH_SIZE = 1024  # Optimal for P100/T4
KAGGLE_EVAL_BATCH_SIZE = 2048  # Larger for inference only


@torch.no_grad()
def gpu_batch_inference(model: nn.Module, X: np.ndarray, device: torch.device, 
                        batch_size: int = KAGGLE_EVAL_BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """
    GPU-optimized batch inference with optional AMP for Kaggle accelerator.
    Returns (predictions, probabilities).
    """
    model.eval()
    n_samples = len(X)
    all_probs = []
    
    for i in range(0, n_samples, batch_size):
        batch_x = torch.tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
        
        if USE_AMP and device.type == 'cuda':
            with autocast():
                logits = model(batch_x)
                probs = F.softmax(logits.float(), dim=-1)  # Cast to float32 for softmax
        else:
            logits = model(batch_x)
            probs = F.softmax(logits, dim=-1)
        
        all_probs.append(probs.cpu().numpy())
    
    probs = np.concatenate(all_probs, axis=0)
    preds = probs.argmax(axis=1)
    return preds, probs


def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error - measures prediction confidence calibration."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        # Get predictions in this confidence bin
        confidences = probs.max(axis=1)
        in_bin = (confidences > lo) & (confidences <= hi)
        if in_bin.sum() == 0:
            continue
        # Accuracy in bin
        preds_in_bin = probs[in_bin].argmax(axis=1)
        labels_in_bin = labels[in_bin]
        acc_in_bin = (preds_in_bin == labels_in_bin).mean()
        # Average confidence in bin
        conf_in_bin = confidences[in_bin].mean()
        # Weighted by bin size
        ece += (in_bin.sum() / len(probs)) * abs(acc_in_bin - conf_in_bin)
    return ece

# =============================================================================
# CONFIGURATION
# =============================================================================

DEFAULT_MODEL_DIR = "/kaggle/input/trained-model/pytorch/default/9"
DEFAULT_DATA_DIR = "/kaggle/input/cic-ids2017/MachineLearningCVE"
DEFAULT_SCALER_PATH = "/kaggle/input/trained-model/pytorch/default/9/scaler.pkl"
DEFAULT_OUTPUT_DIR = "/kaggle/working"

NUM_SAMPLES = 50
RANDOM_SEED = 42

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# =============================================================================
# MODEL ARCHITECTURES
# =============================================================================

class MLPDropout(nn.Module):
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(64, d_out),
        )
    def forward(self, x): return self.net(x)


class DeepCNN(nn.Module):
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        self.proj = nn.Linear(d_in, 128)
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv1d(128, 64, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, d_out))
    def forward(self, x):
        x = F.relu(self.proj(x)).unsqueeze(1)
        x = F.relu(self.conv1(x)); x = F.relu(self.conv2(x)); x = F.relu(self.conv3(x))
        return self.fc(self.pool(x).squeeze(-1))


class DeepTCN(nn.Module):
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        self.tcn1 = nn.Conv1d(1, 32, 3, dilation=1, padding=1)
        self.tcn2 = nn.Conv1d(32, 64, 3, dilation=2, padding=2)
        self.tcn3 = nn.Conv1d(64, 32, 3, dilation=4, padding=4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, d_out))
    def forward(self, x):
        x = F.relu(self.input(x)).unsqueeze(1)
        x = F.relu(self.tcn1(x)); x = F.relu(self.tcn2(x)); x = F.relu(self.tcn3(x))
        return self.fc(self.pool(x).squeeze(-1))


class BiLSTMAttention(nn.Module):
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(d_in, hidden_size), nn.LayerNorm(hidden_size), nn.Dropout(p))
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True, dropout=p if num_layers > 1 else 0., bidirectional=True)
        self.attention = nn.MultiheadAttention(hidden_size * 2, num_heads, dropout=p, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_size * 2)
        self.classifier = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden_size, n_classes))
    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1)
        lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        return self.classifier(self.attn_norm(lstm_out + attn_out).squeeze(1))


# =============================================================================
# RL AGENT
# =============================================================================

class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.feature = nn.Sequential(nn.Linear(state_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU())
        self.value_stream = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage_stream = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, action_dim))
    def forward(self, x):
        f = self.feature(x)
        v, a = self.value_stream(f), self.advantage_stream(f)
        return v + (a - a.mean(dim=1, keepdim=True))


# =============================================================================
# MODEL SELECTION (RL)
# =============================================================================

class RLSelectEnsemble(nn.Module):
    """Ensemble with RL-guided model selection - with state noise for robustness."""
    
    def __init__(self, models: Dict[str, nn.Module], device: torch.device = None, 
                 q_network: nn.Module = None, selection_temp: float = 0.5, seed: int = None,
                 state_noise_std: float = 0.1):
        super().__init__()
        self.models = nn.ModuleDict(models)
        self.device = device or torch.device("cpu")
        self._q_network = q_network
        self.selection_temp = selection_temp
        self.state_noise_std = state_noise_std  # Noise to confuse attacker
        self.stochastic_selection = True  # Use softmax sampling
        self.last_action = 0
        self.last_def_action, self.last_att_action = 0, 0
        self.last_rewards = (0.0, 0.0)
        self.selection_history = []  # Track which models were selected
        # Seed for reproducibility
        if seed is not None:
            torch.manual_seed(seed)
        self._seed = seed
        self.rng = np.random.RandomState(seed)  # Local RNG for state noise
    
    @property
    def model_list(self):
        return list(self.models.values())
    
    @property
    def model_names(self):
        return list(self.models.keys())
        
    def _build_state(self, x: torch.Tensor) -> torch.Tensor:
        """Build state tensor with random noise to confuse attacker."""
        feat = x.mean(dim=0)  # [feature_dim]
        
        # Add random noise to feature representation - makes state unpredictable
        if self.state_noise_std > 0:
            noise = torch.tensor(
                self.rng.randn(feat.shape[0]) * self.state_noise_std,
                device=x.device, dtype=x.dtype
            )
            feat = feat + noise
        
        def_oh = torch.zeros(12, device=x.device, dtype=x.dtype)
        def_oh[self.last_def_action] = 1.0
        att_oh = torch.zeros(12, device=x.device, dtype=x.dtype)
        att_oh[self.last_att_action] = 1.0
        rewards = torch.tensor(self.last_rewards, device=x.device, dtype=x.dtype)
        
        return torch.cat([feat, def_oh, att_oh, rewards])  # [state_dim]
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """RL-guided model selection - Q-network chooses the best model."""
        num_models = len(self.model_list)
        
        if self._q_network is not None:
            state = self._build_state(x).unsqueeze(0)
            with torch.no_grad():
                q_vals = self._q_network(state).squeeze(0)
            
            if self.stochastic_selection:
                # Softmax sampling with temperature
                probs = F.softmax(q_vals / self.selection_temp, dim=-1)
                action = torch.multinomial(probs, 1).item()
            else:
                # Greedy selection
                action = q_vals.argmax().item()
            
            self.last_action = action
            self.last_def_action = action
        else:
            # Fallback to first model if no Q-network
            action = 0
        
        # Map action to model index (action space = num_models * 3, we use model part)
        model_idx = min(action // 3, num_models - 1)
        self.selection_history.append(model_idx)
        
        selected_model = self.model_list[model_idx]
        selected_model.eval()
        with torch.no_grad():
            logits = selected_model(x)
        
        return logits
    
    def to(self, device):
        """Override to() to also move q_network."""
        super().to(device)
        self.device = device if isinstance(device, torch.device) else torch.device(device)
        if self._q_network is not None:
            self._q_network = self._q_network.to(self.device)
        return self
    
    def get_selection_stats(self) -> Dict[str, int]:
        """Get statistics of model selection."""
        stats = {}
        for i, name in enumerate(self.model_names):
            stats[name] = self.selection_history.count(i)
        return stats
    
    def reset_history(self):
        self.selection_history = []
    
    def reset_seed(self, seed: int = None):
        """Reset torch seed and numpy RNG for reproducibility."""
        if seed is not None:
            torch.manual_seed(seed)
            self.rng = np.random.RandomState(seed)
        self._seed = seed
    
    def set_temperature(self, temp: float):
        """Set selection temperature. Higher = more random, Lower = more greedy."""
        self.selection_temp = temp
    
    def set_state_noise(self, std: float):
        """Set state noise standard deviation. Higher = more unpredictable."""
        self.state_noise_std = std
    
    def enable_stochastic(self): self.stochastic_selection = True
    def disable_stochastic(self): self.stochastic_selection = False


# =============================================================================
# UTILITIES
# =============================================================================

class MutationDefenseClassifier:
    """
    Wrapper class cho hệ thống Mutation Defense.
    Bọc toàn bộ hệ thống defense (RL + Mutation + Multi-arm) thành một black-box classifier.
    ART sẽ tấn công wrapper này như một mô hình duy nhất.
    
    Khi ART gọi predict(), bên trong sẽ thực thi:
    1. RL agent chọn model
    2. 3-arm mutation
    3. Aggregation
    → Trả về prediction cuối cùng
    """
    
    def __init__(self, model: nn.Module, input_shape: Tuple[int, ...], 
                 nb_classes: int, device: torch.device, clip_values: Tuple[float, float] = (-5., 5.)):
        self.model = model
        self.input_shape = input_shape
        self.nb_classes = nb_classes
        self.device = device
        self.clip_values = clip_values
        
    def predict(self, x: np.ndarray, batch_size: int = 32) -> np.ndarray:
        """
        Predict probabilities - interface for ART.
        Input: numpy array [batch, features]
        Output: numpy array [batch, nb_classes] với probabilities
        """
        preds = []
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(x), batch_size):
                batch = torch.tensor(x[i:i+batch_size], dtype=torch.float32, device=self.device)
                logits = self.model(batch)
                probs = F.softmax(logits, dim=-1).cpu().numpy()
                preds.append(probs)
        return np.vstack(preds)
    
    def predict_classes(self, x: np.ndarray, batch_size: int = 32) -> np.ndarray:
        """Predict class labels."""
        probs = self.predict(x, batch_size)
        return np.argmax(probs, axis=1)


def load_attack_samples(data_dir, scaler_path, num_samples=100, seed=42):
    csv_files = ["Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
                 "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv", "Tuesday-WorkingHours.pcap_ISCX.csv"]
    drop_cols = ["Flow ID", "Fwd Header Length.1", "Source IP", "Src IP", "Source Port", "Src Port",
                 "Destination IP", "Dst IP", "Destination Port", "Dst Port", "Timestamp"]
    dfs = []
    for f in csv_files:
        path = os.path.join(data_dir, f)
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            df.drop(columns=drop_cols, inplace=True, errors="ignore")
            df["Label"] = df["Label"].replace({"BENIGN": "Benign"})
            df = df[df["Label"] != "Benign"]
            dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_all.dropna(inplace=True)
    np.random.seed(seed)
    if len(df_all) > num_samples:
        df_all = df_all.sample(n=num_samples, random_state=seed)
    X = df_all.drop("Label", axis=1).values.astype(np.float32)
    y = np.ones(len(X), dtype=np.int64)
    if os.path.exists(scaler_path):
        X = joblib.load(scaler_path).transform(X).astype(np.float32)
    logger.info(f"Loaded {len(X)} samples, {X.shape[1]} features")
    return X, y


def load_model_pool(model_dir, input_dim, device):
    specs = [("mlp", MLPDropout), ("cnn", DeepCNN), ("tcn", DeepTCN), ("bilstm", BiLSTMAttention)]
    pool = {}
    for name, cls in specs:
        model = cls(input_dim, 2).to(device)
        loaded = False
        for suffix in ["_model_final.pth", "_model.pth"]:
            path = os.path.join(model_dir, name + suffix)
            if os.path.exists(path):
                ckpt = torch.load(path, map_location=device)
                model.load_state_dict(ckpt.get("state_dict", ckpt))
                logger.info(f"Loaded {name} from {path}")
                loaded = True
                break
        if not loaded:
            logger.warning(f"Model {name} not found in {model_dir}, using random init!")
        model.eval()
        pool[name] = model
    return pool


def verify_model_pool(model_pool, X, y, device):
    """Verify each model's individual performance before defense."""
    logger.info("\n" + "="*60)
    logger.info("MODEL POOL VERIFICATION (no mutations)")
    logger.info("="*60)
    
    best_model = None
    best_acc = 0
    
    for name, model in model_pool.items():
        model.eval()
        x_tensor = torch.tensor(X, dtype=torch.float32, device=device)
        with torch.no_grad():
            logits = model(x_tensor)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=1)
        
        acc = accuracy_score(y, preds)
        dr = recall_score(y, preds, average="binary", zero_division=0)
        
        # Check prediction distribution
        pred_0 = (preds == 0).sum()
        pred_1 = (preds == 1).sum()
        
        logger.info(f"  {name}: Acc={acc:.4f}, DR={dr:.4f}, Pred[0]={pred_0}, Pred[1]={pred_1}")
        
        if acc > best_acc:
            best_acc = acc
            best_model = name
    
    logger.info(f"  Best model: {best_model} with Acc={best_acc:.4f}")
    return best_model, best_acc


def load_q_network(model_dir, output_dir, state_dim, device):
    q = DuelingQNetwork(state_dim, 12).to(device)
    for path in [os.path.join(model_dir, "defender_latest.pth"), os.path.join(output_dir, "defender_latest.pth")]:
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=device)
            q.load_state_dict(ckpt.get("state_dict", ckpt))
            logger.info(f"Loaded defender from {path}")
            return q, True
    return q, False


def evaluate(classifier, X, y, name):
    """
    Evaluate classifier with full metrics: Acc, DR, F1, AUC, ECE.
    
    For adversarial evaluation (y=1 for all samples):
    - Detection Rate (DR) = % samples correctly predicted as attack (pred=1)
    - Evasion Rate = 1 - DR = % samples fooled to benign (pred=0)
    """
    probs = classifier.predict(X)
    preds = np.argmax(probs, axis=1)
    
    acc = accuracy_score(y, preds)
    dr = recall_score(y, preds, average="binary", zero_division=0)  # Detection Rate = Recall
    f1 = f1_score(y, preds, average="binary", zero_division=0)
    
    # AUC - use probability of positive class
    # NOTE: AUC=0.5 is expected when y contains only one class (all attacks)
    # This is not a bug - we need both benign and attack samples for meaningful AUC
    try:
        if len(np.unique(y)) < 2:
            auc = np.nan  # Mark as N/A when only one class
        else:
            auc = roc_auc_score(y, probs[:, 1])
    except ValueError:
        auc = np.nan  # Default if only one class present
    
    # ECE - Expected Calibration Error
    ece = compute_ece(probs, y)
    
    # Evasion rate = 1 - DR (for attack samples where y=1)
    evasion_rate = 1.0 - dr
    
    logger.info(f"{name}: Acc={acc:.4f}, DR={dr:.4f}, F1={f1:.4f}, AUC={auc:.4f}, ECE={ece:.4f}, Evasion={evasion_rate:.2%}")
    return {"accuracy": acc, "detection_rate": dr, "f1_score": f1, "auc": auc, "ece": ece, "evasion_rate": evasion_rate}


# =============================================================================
# MAIN EVALUATION - RL SELECT (Clean Data Only)
# =============================================================================

def run_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")
    
    X, y = load_attack_samples(DEFAULT_DATA_DIR, DEFAULT_SCALER_PATH, NUM_SAMPLES)
    input_dim = X.shape[1]
    model_pool = load_model_pool(DEFAULT_MODEL_DIR, input_dim, device)
    
    state_dim = input_dim + 12 + 12 + 2
    logger.info(f"Input dim: {input_dim}, State dim: {state_dim}")
    q_net, loaded = load_q_network(DEFAULT_MODEL_DIR, DEFAULT_OUTPUT_DIR, state_dim, device)
    if not loaded: 
        logger.warning("Q-Network not found, RL Select will fallback to first model")
        q_net = None
    
    # Test multiple noise levels for state perturbation
    RL_NOISE_STDS = [0.0, 0.1, 0.5, 1.0]  # 0.0 = deterministic baseline
    DEFAULT_TEMP = 1.0  # Fixed temperature, vary noise instead
    
    results = {}
    for noise in RL_NOISE_STDS:
        results[f"rl_noise_{noise}"] = {}
    
    # Verify model pool first
    best_model, best_acc = verify_model_pool(model_pool, X, y, device)
    
    # ========================================================================
    # RL SELECT ENSEMBLE - Multiple Noise Levels
    # ========================================================================
    rl_ensembles = {}
    rl_classifiers = {}
    
    test_x = torch.tensor(X[:2], dtype=torch.float32, device=device)
    
    for noise in RL_NOISE_STDS:
        logger.info("\n" + "=" * 60)
        logger.info(f"RL SELECT ENSEMBLE (noise_std={noise})")
        logger.info("=" * 60)
        
        rl_ens = RLSelectEnsemble(model_pool, device, q_net, selection_temp=DEFAULT_TEMP, 
                                   state_noise_std=noise, seed=RANDOM_SEED).to(device)
        rl_ens.eval()
        rl_ens.enable_stochastic()
        rl_ens.reset_history()
        rl_ens.reset_seed(RANDOM_SEED)
        
        # Test forward pass
        with torch.no_grad():
            test_out = rl_ens(test_x)
            logger.info(f"RL (noise={noise}) output shape: {test_out.shape}")
        
        rl_cls = MutationDefenseClassifier(rl_ens, (input_dim,), 2, device)
        
        # Reset seed and evaluate clean
        rl_ens.reset_seed(RANDOM_SEED)
        rl_ens.reset_history()
        results[f"rl_noise_{noise}"]["clean"] = evaluate(rl_cls, X, y, f"RL(n={noise}) Clean")
        logger.info(f"RL (noise={noise}) model usage: {rl_ens.get_selection_stats()}")
        
        rl_ensembles[noise] = rl_ens
        rl_classifiers[noise] = rl_cls
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    # ========================================================================
    # SUMMARY
    # ========================================================================
    logger.info("\n" + "=" * 80)
    logger.info("SUMMARY - RL SELECT (Multiple Noise Levels)")
    logger.info("=" * 80)
    
    print("\n" + "=" * 80)
    print("📊 RL SELECT EVALUATION (Clean Data)")
    print("=" * 80)
    
    metrics = ["accuracy", "detection_rate", "f1_score", "ece"]
    metric_names = ["Acc", "DR", "F1", "ECE"]
    
    # Build method list
    methods = [f"rl_noise_{n}" for n in RL_NOISE_STDS]
    method_names = [f"RL(σ={n})" for n in RL_NOISE_STDS]
    
    # Header
    header = f"{'Metric':<10}"
    for name in method_names:
        header += f" {name:<12}"
    print(header)
    print("-" * (10 + 13 * len(method_names)))
    
    for metric, mname in zip(metrics, metric_names):
        row = f"{mname:<10}"
        for method in methods:
            res = results.get(method, {}).get("clean", {})
            val = res.get(metric, None)
            if val is None or (isinstance(val, float) and np.isnan(val)):
                row += f" {'N/A':<12}"
            else:
                row += f" {val:.4f}{'':>6}"
        print(row)
    
    # Best noise level
    print("\n" + "-" * 80)
    best_dr = 0
    best_noise = None
    for noise in RL_NOISE_STDS:
        dr = results.get(f"rl_noise_{noise}", {}).get("clean", {}).get("detection_rate", 0)
        if dr > best_dr:
            best_dr = dr
            best_noise = noise
    print(f"🏆 Best noise level: σ={best_noise} with DR={best_dr:.4f}")
    
    # Sync CUDA before timing
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    # ========================================================================
    # INFERENCE TIMING
    # ========================================================================
    logger.info("\nINFERENCE TIMING:")
    x_test = torch.tensor(X[:10], dtype=torch.float32, device=device)
    
    # Warmup
    with torch.no_grad(): 
        for rl_ens in rl_ensembles.values():
            rl_ens(x_test)
    
    # Time RL Select (use first noise level as reference)
    rl_ens_ref = list(rl_ensembles.values())[0]
    start = time.time()
    for _ in range(100):
        with torch.no_grad(): rl_ens_ref(x_test)
    rl_t = (time.time() - start) / 100 * 1000
    logger.info(f"  RL Select: {rl_t:.2f} ms")
    
    # Save results
    with open(os.path.join(DEFAULT_OUTPUT_DIR, "rl_eval_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    
    return results


# =============================================================================
# DRIFT EVALUATION - CONFIGURATION
# =============================================================================

@dataclass
class DriftConfig:
    """Configuration for drift experiments."""
    drift_type: str = "temporal"  # temporal (real), sudden, gradual, incremental, recurring, mixed
    drift_severity: float = 1.0  # For synthetic drift injection
    drift_point: int = 5000  # Sample index where drift starts (for synthetic)
    transition_window: int = 1000  # For gradual/incremental drift
    period: int = 2000  # For recurring drift
    recovery_threshold: float = 0.90  # 90% of baseline = "recovered"
    eval_window: int = 100  # Window size for evaluation
    seed: int = 42
    # Temporal drift settings (CIC-IDS2017)
    train_days: List[str] = None  # Days for training (default: Monday)
    test_days: List[str] = None   # Days for testing with natural drift


@dataclass 
class DriftMetrics:
    """Comprehensive metrics for drift adaptation evaluation."""
    detection_delay: int = -1  # Steps from drift to detection
    false_alarm_rate: float = 0.0
    recovery_time: int = -1  # Steps to recover 90% baseline
    recovery_rate: float = 0.0  # Accuracy gain per 1000 steps
    accuracy_drop: float = 0.0  # Max accuracy decrease
    final_accuracy: float = 0.0
    prequential_accuracy: float = 0.0  # Online accuracy with fading
    stability_index: float = 0.0  # Variance of accuracy
    total_errors_during_drift: int = 0
    # Additional metrics for Q1 journal
    f1_before_drift: float = 0.0
    f1_after_drift: float = 0.0
    auc_before_drift: float = 0.0
    auc_after_drift: float = 0.0
    detection_rate_drop: float = 0.0


# =============================================================================
# TEMPORAL DATA LOADING - REAL DRIFT FROM CIC-IDS2017 DAYS
# =============================================================================

# CIC-IDS2017 files by day - each day has different attack patterns (natural drift)
CIC_IDS2017_DAYS = {
    "Monday": ["Monday-WorkingHours.pcap_ISCX.csv"],  # Mostly benign (baseline)
    "Tuesday": ["Tuesday-WorkingHours.pcap_ISCX.csv"],  # FTP-Patator, SSH-Patator
    "Wednesday": ["Wednesday-workingHours.pcap_ISCX.csv"],  # DoS Slowloris, DoS Slowhttptest, DoS Hulk, DoS GoldenEye, Heartbleed
    "Thursday": [
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",  # Web Attack – Brute Force, XSS, SQL Injection
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv"  # Infiltration
    ],
    "Friday": [
        "Friday-WorkingHours-Morning.pcap_ISCX.csv",  # Bot
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",  # PortScan
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"  # DDoS
    ]
}

# Attack types per day for documentation
DAY_ATTACKS = {
    "Monday": ["Benign (baseline)"],
    "Tuesday": ["FTP-Patator", "SSH-Patator"],
    "Wednesday": ["DoS Slowloris", "DoS Slowhttptest", "DoS Hulk", "DoS GoldenEye", "Heartbleed"],
    "Thursday": ["Web Attack Brute Force", "Web Attack XSS", "Web Attack SQL Injection", "Infiltration"],
    "Friday": ["Bot", "PortScan", "DDoS"]
}


def load_day_data(data_dir: str, day: str, scaler_path: str, max_samples_per_file: int = None, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Load data for a specific day from CIC-IDS2017.
    
    Returns:
        X: Features array
        y: Labels (0=Benign, 1=Attack)
        attack_types: List of attack types present
    """
    drop_cols = ["Flow ID", "Fwd Header Length.1", "Source IP", "Src IP", "Source Port", "Src Port",
                 "Destination IP", "Dst IP", "Destination Port", "Dst Port", "Timestamp"]
    
    files = CIC_IDS2017_DAYS.get(day, [])
    dfs = []
    attack_types = set()
    
    for f in files:
        path = os.path.join(data_dir, f)
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            df.drop(columns=drop_cols, inplace=True, errors="ignore")
            df["Label"] = df["Label"].replace({"BENIGN": "Benign"})
            
            # Track attack types
            for label in df["Label"].unique():
                if label != "Benign":
                    attack_types.add(label)
            
            dfs.append(df)
    
    if not dfs:
        logger.warning(f"No data files found for {day}")
        return np.array([]), np.array([]), []
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_all.dropna(inplace=True)
    
    # Sample if too large
    np.random.seed(seed)
    if max_samples_per_file and len(df_all) > max_samples_per_file * len(files):
        df_all = df_all.sample(n=max_samples_per_file * len(files), random_state=seed)
    
    X = df_all.drop("Label", axis=1).values.astype(np.float32)
    y = (df_all["Label"] != "Benign").astype(np.int64).values
    
    # Apply scaler
    if os.path.exists(scaler_path):
        X = joblib.load(scaler_path).transform(X).astype(np.float32)
    
    return X, y, list(attack_types)


def load_temporal_drift_data(
    data_dir: str, 
    scaler_path: str, 
    train_days: List[str] = None,
    test_days: List[str] = None,
    max_samples_per_day: int = 10000,
    seed: int = 42
) -> Dict:
    """
    Load CIC-IDS2017 data with temporal split for real drift evaluation.
    
    Default: Train on Monday (baseline), test on Tuesday→Friday (progressive drift)
    
    Returns:
        data_dict: {
            "train": (X_train, y_train, attack_types),
            "test_days": [(day_name, X, y, attack_types), ...]
        }
    """
    train_days = train_days or ["Monday"]
    test_days = test_days or ["Tuesday", "Wednesday", "Thursday", "Friday"]
    
    logger.info(f"Loading temporal drift data: Train={train_days}, Test={test_days}")
    
    # Load training data
    train_X_list, train_y_list = [], []
    train_attacks = []
    for day in train_days:
        X, y, attacks = load_day_data(data_dir, day, scaler_path, max_samples_per_day, seed)
        if len(X) > 0:
            train_X_list.append(X)
            train_y_list.append(y)
            train_attacks.extend(attacks)
            logger.info(f"  {day}: {len(X)} samples, Attack ratio={y.mean():.2%}, Attacks={attacks}")
    
    X_train = np.vstack(train_X_list) if train_X_list else np.array([])
    y_train = np.concatenate(train_y_list) if train_y_list else np.array([])
    
    # Load test data per day (for temporal analysis)
    test_data = []
    for day in test_days:
        X, y, attacks = load_day_data(data_dir, day, scaler_path, max_samples_per_day, seed)
        if len(X) > 0:
            test_data.append((day, X, y, attacks))
            logger.info(f"  {day}: {len(X)} samples, Attack ratio={y.mean():.2%}, Attacks={attacks}")
    
    return {
        "train": (X_train, y_train, train_attacks),
        "test_days": test_data
    }


# =============================================================================
# DRIFT INJECTOR - Inject controlled drift into real data
# =============================================================================

class DriftInjector:
    """
    Controlled drift injection for reproducible experiments.
    
    Drift types (per Lu et al., TKDE 2018):
    1. Covariate drift: P(X) changes
    2. Prior drift: P(Y) changes  
    3. Conditional drift: P(Y|X) changes
    4. Mixed drift: All above combined
    """
    
    def __init__(self, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.important_features = list(range(10))  # First 10 features
    
    def inject_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Main entry point for drift injection."""
        if config.drift_type == "sudden":
            return self._inject_sudden_covariate_drift(X, y, config)
        elif config.drift_type == "gradual":
            return self._inject_gradual_drift(X, y, config)
        elif config.drift_type == "incremental":
            return self._inject_incremental_drift(X, y, config)
        elif config.drift_type == "recurring":
            return self._inject_recurring_drift(X, y, config)
        elif config.drift_type == "prior":
            return self._inject_prior_drift(X, y, config)
        elif config.drift_type == "conditional":
            return self._inject_conditional_drift(X, y, config)
        elif config.drift_type == "mixed":
            return self._inject_mixed_drift(X, y, config)
        else:
            raise ValueError(f"Unknown drift type: {config.drift_type}")
    
    def _inject_sudden_covariate_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Sudden covariate drift: P(X) changes instantly. Simulates network reconfiguration."""
        X_drifted = X.copy()
        drift_point = min(config.drift_point, len(X) - 1)
        severity = config.drift_severity
        
        ref_std = X[:drift_point].std(axis=0) + 1e-8
        for feat_idx in self.important_features:
            if feat_idx < X.shape[1]:
                shift = severity * ref_std[feat_idx] * self.rng.choice([-1, 1])
                X_drifted[drift_point:, feat_idx] += shift
        
        return X_drifted, y.copy()
    
    def _inject_gradual_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Gradual drift: Smooth transition over window. Simulates seasonal changes."""
        X_drifted = X.copy()
        start = min(config.drift_point, len(X) - 1)
        end = min(start + config.transition_window, len(X))
        severity = config.drift_severity
        
        ref_std = X[:start].std(axis=0) + 1e-8
        for feat_idx in self.important_features:
            if feat_idx < X.shape[1]:
                target_shift = severity * ref_std[feat_idx] * self.rng.choice([-1, 1])
                for i, idx in enumerate(range(start, end)):
                    progress = i / max(1, config.transition_window)
                    X_drifted[idx, feat_idx] += target_shift * progress
                if end < len(X):
                    X_drifted[end:, feat_idx] += target_shift
        
        return X_drifted, y.copy()
    
    def _inject_incremental_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Incremental drift: Continuous slow shift. Simulates gradual network degradation."""
        X_drifted = X.copy()
        start = min(config.drift_point, len(X) - 1)
        severity = config.drift_severity
        
        ref_std = X[:start].std(axis=0) + 1e-8
        total_samples = max(1, len(X) - start)
        
        for feat_idx in self.important_features:
            if feat_idx < X.shape[1]:
                final_shift = severity * ref_std[feat_idx] * self.rng.choice([-1, 1])
                for i, idx in enumerate(range(start, len(X))):
                    progress = i / total_samples
                    X_drifted[idx, feat_idx] += final_shift * progress
        
        return X_drifted, y.copy()
    
    def _inject_recurring_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Recurring drift: Periodic oscillation. Simulates day/night patterns."""
        X_drifted = X.copy()
        start = min(config.drift_point, len(X) - 1)
        period = max(1, config.period)
        severity = config.drift_severity
        
        ref_std = X[:start].std(axis=0) + 1e-8
        for feat_idx in self.important_features:
            if feat_idx < X.shape[1]:
                max_shift = severity * ref_std[feat_idx]
                for idx in range(start, len(X)):
                    phase = 2 * np.pi * (idx - start) / period
                    current_shift = max_shift * np.sin(phase)
                    X_drifted[idx, feat_idx] += current_shift
        
        return X_drifted, y.copy()
    
    def _inject_prior_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Prior drift: P(Y) changes. Simulates attack campaign starts."""
        y_drifted = y.copy()
        drift_point = min(config.drift_point, len(y) - 1)
        new_attack_ratio = min(0.8, 0.2 + config.drift_severity * 0.2)
        
        post_drift_indices = np.arange(drift_point, len(y))
        if len(post_drift_indices) == 0:
            return X.copy(), y_drifted
        
        current_attack_ratio = y[post_drift_indices].mean()
        
        if new_attack_ratio > current_attack_ratio:
            benign_mask = y[post_drift_indices] == 0
            benign_indices = post_drift_indices[benign_mask]
            if len(benign_indices) > 0:
                flip_count = int((new_attack_ratio - current_attack_ratio) * len(post_drift_indices))
                flip_indices = self.rng.choice(benign_indices, min(flip_count, len(benign_indices)), replace=False)
                y_drifted[flip_indices] = 1
        else:
            attack_mask = y[post_drift_indices] == 1
            attack_indices = post_drift_indices[attack_mask]
            if len(attack_indices) > 0:
                flip_count = int((current_attack_ratio - new_attack_ratio) * len(post_drift_indices))
                flip_indices = self.rng.choice(attack_indices, min(flip_count, len(attack_indices)), replace=False)
                y_drifted[flip_indices] = 0
        
        return X.copy(), y_drifted
    
    def _inject_conditional_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Conditional drift: P(Y|X) changes (true concept drift). Most challenging for IDS."""
        y_drifted = y.copy()
        drift_point = min(config.drift_point, len(y) - 1)
        flip_rate = min(0.3, config.drift_severity * 0.1)
        
        post_drift_indices = np.arange(drift_point, len(y))
        flip_mask = self.rng.rand(len(post_drift_indices)) < flip_rate
        flip_indices = post_drift_indices[flip_mask]
        y_drifted[flip_indices] = 1 - y_drifted[flip_indices]
        
        return X.copy(), y_drifted
    
    def _inject_mixed_drift(self, X: np.ndarray, y: np.ndarray, config: DriftConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Mixed drift: Combination of all types. Simulates major network incident."""
        X_drifted, _ = self._inject_sudden_covariate_drift(X, y, config)
        
        config_prior = DriftConfig(drift_point=config.drift_point, drift_severity=config.drift_severity * 0.5)
        _, y_drifted = self._inject_prior_drift(X_drifted, y, config_prior)
        
        config_cond = DriftConfig(drift_point=config.drift_point, drift_severity=config.drift_severity * 0.3)
        _, y_drifted = self._inject_conditional_drift(X_drifted, y_drifted, config_cond)
        
        return X_drifted, y_drifted


# =============================================================================
# DRIFT DETECTION BASELINES
# =============================================================================

class DDM:
    """Drift Detection Method (Gama et al., SBIA 2004). More sensitive version."""
    
    def __init__(self, warning_level: float = 1.5, drift_level: float = 2.0, min_instances: int = 20):
        self.warning_level = warning_level
        self.drift_level = drift_level
        self.min_instances = min_instances
        self.reset()
    
    def reset(self):
        self.n = 0
        self.p = 0.0
        self.s = 0.0
        self.p_min = float('inf')
        self.s_min = float('inf')
        self.drift_detected = False
    
    def update(self, error: float) -> Dict[str, bool]:
        self.n += 1
        self.p = self.p + (error - self.p) / self.n
        self.s = np.sqrt(self.p * (1 - self.p) / self.n) if self.n > 0 else 0.0
        
        if self.n < self.min_instances:
            return {"drift": False, "warning": False}
        
        if self.p + self.s < self.p_min + self.s_min:
            self.p_min = self.p
            self.s_min = self.s
        
        if self.p + self.s >= self.p_min + self.drift_level * self.s_min:
            self.drift_detected = True
            return {"drift": True, "warning": True}
        
        if self.p + self.s >= self.p_min + self.warning_level * self.s_min:
            return {"drift": False, "warning": True}
        
        return {"drift": False, "warning": False}


class ADWIN:
    """
    ADaptive WINdowing (Bifet & Gavaldà, SDM 2007).
    TUNED for better sensitivity on temporal drift scenarios.
    
    Changes from original:
    - delta=0.05 (more sensitive to drift)
    - Smaller window for faster detection
    - Adaptive epsilon scaling
    """
    
    def __init__(self, delta: float = 0.05):  # TUNED: Higher delta = more sensitive
        self.delta = delta
        self.window = deque(maxlen=1000)  # TUNED: Smaller window = faster detection
        self.drift_detected = False
        self.warning_count = 0
    
    def reset(self):
        self.window.clear()
        self.drift_detected = False
        self.warning_count = 0
    
    def update(self, value: float) -> Dict[str, bool]:
        self.window.append(value)
        
        if len(self.window) < 30:  # TUNED: Reduced min window
            return {"drift": False, "warning": False}
        
        n = len(self.window)
        window_list = list(self.window)
        
        # TUNED: More aggressive split point checking
        for split_point in range(20, n - 20, 5):  # Smaller step = more checks
            w0 = window_list[:split_point]
            w1 = window_list[split_point:]
            m0, m1 = np.mean(w0), np.mean(w1)
            n0, n1 = len(w0), len(w1)
            
            # Hoeffding bound with adaptive scaling
            m = 1.0 / (1.0/n0 + 1.0/n1)
            eps_cut = np.sqrt(np.log(2.0/self.delta) / (2.0 * m))
            
            # TUNED: Two-level detection
            diff = abs(m0 - m1)
            
            if diff > eps_cut:
                # Clear drift detected
                for _ in range(split_point):
                    if self.window:
                        self.window.popleft()
                self.drift_detected = True
                self.warning_count = 0
                return {"drift": True, "warning": False}
            
            elif diff > eps_cut * 0.7:  # Warning level
                self.warning_count += 1
                if self.warning_count >= 3:  # Consecutive warnings → drift
                    for _ in range(split_point // 2):
                        if self.window:
                            self.window.popleft()
                    self.drift_detected = True
                    self.warning_count = 0
                    return {"drift": True, "warning": True}
                return {"drift": False, "warning": True}
        
        self.warning_count = 0
        return {"drift": False, "warning": False}


class PageHinkley:
    """Page-Hinkley Test (Page, Biometrika 1954) - CUSUM-based drift detection."""
    
    def __init__(self, delta: float = 0.005, lambda_: float = 50.0, alpha: float = 0.9999):
        self.delta = delta
        self.lambda_ = lambda_
        self.alpha = alpha
        self.reset()
    
    def reset(self):
        self.n = 0
        self.sum = 0.0
        self.x_mean = 0.0
        self.min_sum = float('inf')
        self.drift_detected = False
    
    def update(self, x: float) -> Dict[str, bool]:
        self.n += 1
        self.x_mean = self.x_mean + (x - self.x_mean) / self.n
        self.sum = self.alpha * self.sum + (x - self.x_mean - self.delta)
        self.min_sum = min(self.min_sum, self.sum)
        
        if self.sum - self.min_sum > self.lambda_:
            self.drift_detected = True
            return {"drift": True, "warning": False}
        
        return {"drift": False, "warning": False}


class KSWIN:
    """
    Kolmogorov-Smirnov Windowing (Raab et al., IJCNN 2020).
    Statistical test comparing two windows using KS test.
    """
    
    def __init__(self, alpha: float = 0.005, window_size: int = 100, stat_size: int = 30):
        self.alpha = alpha
        self.window_size = window_size
        self.stat_size = stat_size
        self.window = deque(maxlen=window_size)
        self.drift_detected = False
    
    def reset(self):
        self.window.clear()
        self.drift_detected = False
    
    def update(self, value: float) -> Dict[str, bool]:
        self.window.append(value)
        
        if len(self.window) < self.window_size:
            return {"drift": False, "warning": False}
        
        # Compare first stat_size samples with last stat_size samples
        w0 = list(self.window)[:self.stat_size]
        w1 = list(self.window)[-self.stat_size:]
        
        # KS test (ks_2samp already imported at module level)
        try:
            stat, p_value = ks_2samp(w0, w1)
            if p_value < self.alpha:
                self.drift_detected = True
                # Slide window by removing first half
                for _ in range(self.window_size // 2):
                    if self.window:
                        self.window.popleft()
                return {"drift": True, "warning": False}
        except:
            pass
        
        return {"drift": False, "warning": False}


# =============================================================================
# SOTA DRIFT EMBEDDING & ADVANCED COMPONENTS (from ids_script.py)
# =============================================================================

@dataclass
class SOTADriftConfig:
    """Configuration for SOTA Drift Embedding v2.3."""
    ref_window: int = 1000
    recent_window: int = 200
    use_frechet: bool = True
    use_wasserstein: bool = True
    use_prediction_entropy: bool = True
    covariate_sensitivity: float = 2.0
    prior_sensitivity: float = 3.0
    conditional_sensitivity: float = 5.0
    # Poisoning detection (v2.3)
    reward_anomaly_threshold: float = 3.0
    poisoning_window: int = 50


class DistanceMetrics:
    """Statistical distance metrics for drift detection."""
    
    @staticmethod
    def frechet_distance(mu1: np.ndarray, sigma1: np.ndarray, 
                         mu2: np.ndarray, sigma2: np.ndarray) -> float:
        """
        Frechet distance (FID) between two Gaussians.
        
        Reference: "GANs Trained by a Two Time-Scale Update Rule" [Heusel et al., NeurIPS 2017]
        """
        diff = mu1 - mu2
        
        # Handle 1D case
        if sigma1.ndim == 0 or sigma2.ndim == 0:
            return float(np.sum(diff ** 2) + sigma1 + sigma2 - 2 * np.sqrt(sigma1 * sigma2))
        
        # Add small epsilon for numerical stability
        sigma1 = sigma1 + np.eye(sigma1.shape[0]) * 1e-6
        sigma2 = sigma2 + np.eye(sigma2.shape[0]) * 1e-6
        
        if sqrtm is not None:
            try:
                covmean = sqrtm(sigma1 @ sigma2)
                if np.iscomplexobj(covmean):
                    covmean = covmean.real
                return float(np.sum(diff ** 2) + np.trace(sigma1 + sigma2 - 2 * covmean))
            except:
                # Fallback: simple trace difference
                return float(np.sum(diff ** 2) + np.trace(sigma1) + np.trace(sigma2))
        else:
            return float(np.sum(diff ** 2) + np.trace(sigma1) + np.trace(sigma2))
    
    @staticmethod
    def wasserstein_1d(x: np.ndarray, y: np.ndarray) -> float:
        """1D Wasserstein (Earth Mover's) distance."""
        return wasserstein_distance(x, y)
    
    @staticmethod
    def js_divergence(p: np.ndarray, q: np.ndarray, n_bins: int = 50) -> float:
        """Jensen-Shannon divergence (symmetric KL)."""
        # Create histograms for comparison
        min_val = min(p.min(), q.min())
        max_val = max(p.max(), q.max())
        bins = np.linspace(min_val, max_val, n_bins + 1)
        
        p_hist, _ = np.histogram(p, bins=bins, density=True)
        q_hist, _ = np.histogram(q, bins=bins, density=True)
        
        # Add small epsilon to avoid log(0)
        p_hist = p_hist + 1e-10
        q_hist = q_hist + 1e-10
        
        # Normalize to probability
        p_hist = p_hist / p_hist.sum()
        q_hist = q_hist / q_hist.sum()
        
        return float(jensenshannon(p_hist, q_hist) ** 2)  # Squared to get divergence


class SOTADriftEmbedding:
    """
    SOTA Drift Embedding v2.3: 20D drift vector with poisoning detection.
    
    Dimensions:
    [0-3]: Distribution distances (Frechet, Wasserstein, Prediction Entropy, JS)
    [4-6]: Drift types (Covariate, Prior, Conditional)
    [7-9]: Temporal dynamics (Velocity, Acceleration, Trend)
    [10-12]: Agreement metrics (Correlation, Consistency, Overall)
    [13-15]: Uncertainty & severity (Entropy, Worst-case, Confidence)
    [16-19]: Poisoning features (Reward anomaly, Pred inconsistency, Temporal instability, Boundary)
    """
    
    def __init__(self, n_features: int, cfg: Optional[SOTADriftConfig] = None):
        self.d = n_features
        self.cfg = cfg or SOTADriftConfig()
        self.metrics = DistanceMetrics()
        
        # Data windows
        self.ref_X = deque(maxlen=self.cfg.ref_window)
        self.recent_X = deque(maxlen=self.cfg.recent_window)
        self.ref_labels = deque(maxlen=self.cfg.ref_window)
        self.recent_labels = deque(maxlen=self.cfg.recent_window)
        self.ref_scores = deque(maxlen=self.cfg.ref_window)
        self.recent_scores = deque(maxlen=self.cfg.recent_window)
        
        # Temporal tracking
        self.drift_history = deque(maxlen=20)
        
        # Poisoning detection
        self.reward_history = deque(maxlen=self.cfg.poisoning_window)
        self.prediction_history = deque(maxlen=self.cfg.poisoning_window)
        self.state_history = deque(maxlen=self.cfg.poisoning_window)
    
    def push(self, x: np.ndarray, score: Optional[float] = None, 
             y: Optional[int] = None, to_ref: bool = False):
        """Add sample to reference or recent window."""
        target_X = self.ref_X if to_ref else self.recent_X
        target_labels = self.ref_labels if to_ref else self.recent_labels
        target_scores = self.ref_scores if to_ref else self.recent_scores
        
        target_X.append(x.flatten()[:self.d])
        if y is not None:
            target_labels.append(y)
        if score is not None:
            target_scores.append(score)
            self.prediction_history.append(score)
    
    def push_reward(self, reward: float):
        """Track reward for anomaly detection."""
        self.reward_history.append(reward)
    
    def push_state_vector(self, state: np.ndarray):
        """Track state for temporal instability detection."""
        self.state_history.append(state.copy())
    
    def push_transition(self, state: np.ndarray, reward: float, 
                        x: np.ndarray, score: float, y: int):
        """Push complete transition data."""
        self.push(x, score, y, to_ref=False)
        self.push_reward(reward)
        self.push_state_vector(state)
    
    def _compute_poisoning_features(self) -> np.ndarray:
        """Compute 4D poisoning detection vector."""
        poisoning_vec = np.zeros(4, dtype=np.float32)
        
        # [16] Reward Anomaly
        if len(self.reward_history) >= 10:
            rewards = np.array(list(self.reward_history))
            mean_r = rewards[:-1].mean() if len(rewards) > 1 else rewards.mean()
            std_r = rewards[:-1].std() if len(rewards) > 1 else 1.0
            
            if std_r > 1e-6:
                recent_reward = rewards[-1]
                z_score = abs((recent_reward - mean_r) / std_r)
                poisoning_vec[0] = min(z_score / self.cfg.reward_anomaly_threshold, 1.0)
        
        # [17] Prediction Inconsistency
        if len(self.prediction_history) >= 10:
            preds = np.array(list(self.prediction_history)[-20:])
            pred_variance = preds.var()
            poisoning_vec[1] = min(pred_variance / 0.1, 1.0)
        
        # [18] Temporal Instability
        if len(self.state_history) >= 2:
            states = list(self.state_history)
            distances = [np.linalg.norm(states[i] - states[i-1]) for i in range(1, len(states))]
            
            if distances:
                current_dist = distances[-1]
                mean_dist = np.mean(distances[:-1]) if len(distances) > 1 else current_dist
                std_dist = np.std(distances[:-1]) if len(distances) > 1 else 1.0
                
                if std_dist > 1e-6:
                    instability_z = abs((current_dist - mean_dist) / std_dist)
                    poisoning_vec[2] = min(instability_z / 3.0, 1.0)
        
        # [19] Boundary Exploitation
        if len(self.prediction_history) >= 5:
            preds = np.array(list(self.prediction_history)[-10:])
            if len(preds) > 0:
                confidence = preds.max()
                uncertainty = preds.std()
                boundary_score = confidence * uncertainty
                poisoning_vec[3] = min(boundary_score / 0.5, 1.0)
        
        return poisoning_vec
    
    def vector(self) -> np.ndarray:
        """Compute 20D SOTA drift embedding."""
        vec = np.zeros(16, dtype=np.float32)
        
        # Handle insufficient data
        if len(self.ref_X) < 50 or len(self.recent_X) < 10:
            vec[13] = 1.0  # Max uncertainty
            vec[15] = 0.0  # Zero confidence
            return np.concatenate([vec, np.zeros(4, dtype=np.float32)])
        
        ref = np.array(list(self.ref_X), dtype=np.float32)
        rec = np.array(list(self.recent_X), dtype=np.float32)
        
        # [0-3] Distribution Distances
        if self.cfg.use_frechet:
            try:
                mu_ref, sigma_ref = np.mean(ref, axis=0), np.cov(ref.T)
                mu_rec, sigma_rec = np.mean(rec, axis=0), np.cov(rec.T)
                vec[0] = self.metrics.frechet_distance(mu_ref, sigma_ref, mu_rec, sigma_rec)
                vec[0] = np.clip(vec[0] / 100.0, 0, 10)
            except:
                vec[0] = 0.0
        
        if self.cfg.use_wasserstein:
            w_dists = []
            for j in range(min(10, ref.shape[1])):
                try:
                    w_dists.append(self.metrics.wasserstein_1d(ref[:, j], rec[:, j]))
                except:
                    pass
            vec[1] = np.mean(w_dists) if w_dists else 0.0
            vec[1] = np.clip(vec[1], 0, 5)
        
        # [2] Prediction Entropy
        if self.cfg.use_prediction_entropy and len(self.recent_scores) > 10:
            scores = np.array(list(self.recent_scores), dtype=np.float32)
            p = np.clip(scores, 0.01, 0.99)
            binary_entropy = -p * np.log(p) - (1-p) * np.log(1-p)
            avg_entropy = binary_entropy.mean()
            vec[2] = np.clip(avg_entropy / 0.693 * 3.0, 0, 3)
        
        # [3] Jensen-Shannon Divergence
        js_vals = []
        for j in range(min(10, ref.shape[1])):
            try:
                js_vals.append(self.metrics.js_divergence(ref[:, j], rec[:, j]))
            except:
                pass
        vec[3] = np.mean(js_vals) if js_vals else 0.0
        vec[3] = np.clip(vec[3], 0, 1)
        
        # [4-6] Drift Types
        ks_scores = []
        for j in range(min(self.d, ref.shape[1])):
            try:
                stat, _ = ks_2samp(ref[:, j], rec[:, j])
                ks_scores.append(stat)
            except:
                pass
        vec[4] = np.mean(ks_scores) * self.cfg.covariate_sensitivity if ks_scores else 0.0
        vec[4] = np.clip(vec[4], 0, 1)
        
        if len(self.ref_labels) > 20 and len(self.recent_labels) > 10:
            ref_pos = np.mean(list(self.ref_labels))
            rec_pos = np.mean(list(self.recent_labels))
            vec[5] = abs(ref_pos - rec_pos) * self.cfg.prior_sensitivity
            vec[5] = np.clip(vec[5], 0, 1)
        
        if len(self.ref_scores) > 20 and len(self.recent_scores) > 10:
            score_shift = abs(np.mean(list(self.ref_scores)) - np.mean(list(self.recent_scores)))
            label_factor = 1.0 + vec[5]
            vec[6] = score_shift * label_factor * self.cfg.conditional_sensitivity
            vec[6] = np.clip(vec[6], 0, 2)
        
        # [7-9] Temporal Dynamics
        current_drift = float(np.mean(vec[:4]))
        self.drift_history.append(current_drift)
        
        if len(self.drift_history) >= 3:
            hist = list(self.drift_history)
            vec[7] = (hist[-1] - hist[0]) / len(hist)
            vec[7] = np.clip(vec[7], -1, 1)
            
            if len(hist) >= 4:
                vel_recent = hist[-1] - hist[-2]
                vel_old = hist[-2] - hist[-3]
                vec[8] = vel_recent - vel_old
                vec[8] = np.clip(vec[8], -1, 1)
            
            x_vals = np.arange(len(hist))
            vec[9] = np.polyfit(x_vals, hist, 1)[0]
            vec[9] = np.clip(vec[9], -0.5, 0.5)
        
        # [10-12] Agreement Metrics
        if vec[0] > 0.01 and vec[1] > 0.01:
            vec[10] = min(vec[0] / (vec[1] + 1e-6), vec[1] / (vec[0] + 1e-6))
            vec[10] = np.clip(vec[10], 0, 1)
        
        if vec[2] > 0.01 and vec[3] > 0.01:
            vec[11] = 1.0 - abs((vec[2] / 3.0) - vec[3])
            vec[11] = np.clip(vec[11], 0, 1)
        
        drift_signals = vec[:4] > np.median(vec[:4])
        vec[12] = float(np.mean(drift_signals))
        
        # [13-15] Uncertainty & Severity
        if len(self.recent_scores) > 10:
            scores = np.array(list(self.recent_scores))
            p = np.clip(np.mean(scores), 0.01, 0.99)
            vec[13] = -p * np.log(p) - (1-p) * np.log(1-p)
            vec[13] /= np.log(2)
        
        vec[14] = np.max(vec[:4])
        var = np.var(vec[:4])
        vec[15] = 1.0 / (1.0 + var)
        
        poisoning_vec = self._compute_poisoning_features()
        return np.concatenate([vec, poisoning_vec])
    
    @staticmethod
    def dim() -> int:
        """Dimension of drift+poisoning embedding."""
        return 20
    
    def interpret(self, vec: np.ndarray) -> Dict[str, float]:
        """Interpret drift vector for debugging."""
        return {
            'frechet_distance': float(vec[0]),
            'wasserstein_mean': float(vec[1]),
            'prediction_entropy_dist': float(vec[2]),
            'js_divergence': float(vec[3]),
            'covariate_score': float(vec[4]),
            'prior_score': float(vec[5]),
            'conditional_score': float(vec[6]),
            'velocity': float(vec[7]),
            'acceleration': float(vec[8]),
            'trend': float(vec[9]),
            'metric_correlation': float(vec[10]),
            'consistency': float(vec[11]),
            'overall_agreement': float(vec[12]),
            'prediction_entropy_unc': float(vec[13]),
            'worst_case': float(vec[14]),
            'confidence': float(vec[15]),
            'reward_anomaly': float(vec[16]) if len(vec) > 16 else 0.0,
            'pred_inconsistency': float(vec[17]) if len(vec) > 17 else 0.0,
            'temporal_instability': float(vec[18]) if len(vec) > 18 else 0.0,
            'boundary_exploitation': float(vec[19]) if len(vec) > 19 else 0.0,
        }


class LazySOTADriftEmbedding:
    """Lazy-evaluated drift embedding with caching."""
    
    def __init__(self, base_drift_embedding: SOTADriftEmbedding, update_freq: int = 10):
        self.base = base_drift_embedding
        self.update_freq = update_freq
        self.step_counter = 0
        self.cached_vector = None
    
    def push(self, x, score=None, y=None, to_ref=False):
        self.base.push(x, score, y, to_ref)
    
    def push_reward(self, reward: float):
        self.base.push_reward(reward)
    
    def push_state_vector(self, state: np.ndarray):
        self.base.push_state_vector(state)
    
    def push_transition(self, state: np.ndarray, reward: float, x: np.ndarray, 
                        score: float, y: int):
        self.base.push_transition(state, reward, x, score, y)
    
    def vector(self) -> np.ndarray:
        self.step_counter += 1
        if self.cached_vector is None or self.step_counter % self.update_freq == 0:
            self.cached_vector = self.base.vector()
        return self.cached_vector
    
    def dim(self) -> int:
        return self.base.dim()
    
    def interpret(self, vec: np.ndarray) -> Dict[str, float]:
        return self.base.interpret(vec)


@dataclass
class ADIBlock:
    """Adversarial Drift Injection block."""
    start: int
    duration: int
    kind: str  # 'covariate', 'prior', 'conditional'
    mode: str  # 'sudden', 'gradual', 'recurring', 'incremental'
    severity: float  # [0..1]
    transition: float = 0.2
    poison_rate: float = 0.0
    target_label: Optional[int] = None


class ADIScheduler:
    """ADI v1.5: Simulated FGSM + adaptive shifts + feature tracking."""
    
    def __init__(self, d_features: int, blocks: Optional[List[ADIBlock]] = None, 
                 rng: Optional[np.random.RandomState] = None):
        self.d = d_features
        self.blocks: List[ADIBlock] = blocks or []
        self.rng = rng or np.random.RandomState(1337)
        self.dirs = self._ortho_basis(d_features)
        
        # Feature Importance Tracking (EMA)
        self.feature_importance = np.ones(d_features, dtype=np.float32) / d_features
        self.importance_decay = 0.99
        
        # Attack Success Tracking
        self.attack_history = deque(maxlen=100)
        self.success_rate = 0.5
        self.difficulty_adjustment = 1.0
    
    def add(self, block: ADIBlock):
        self.blocks.append(block)
    
    def active(self, training_step: int) -> Optional[ADIBlock]:
        """Check if any ADI block is active at given training step."""
        for b in self.blocks:
            if b.start <= training_step < (b.start + b.duration):
                return b
        return None
    
    def apply(self, training_step: int, x: np.ndarray, y: Optional[int]) -> Tuple[np.ndarray, Optional[int], Dict[str, float]]:
        """Apply adaptive drift injection."""
        b = self.active(training_step)
        if b is None:
            return x, y, {"severity": 0.0}
        
        p = self._progress(b, training_step)
        sev = b.severity * p * self.difficulty_adjustment
        
        x2, y2 = np.copy(x), (None if y is None else int(y))
        
        if b.kind == 'covariate':
            x2 = self._simulated_fgsm_shift(x2, sev)
        elif b.kind == 'prior':
            y2 = self._prior_shift(y2, sev, b.target_label)
        elif b.kind == 'conditional':
            x2, y2 = self._adaptive_conditional_shift(x2, y2, sev)
        
        if b.poison_rate > 0.0 and self.rng.rand() < b.poison_rate:
            y2 = self._poison_label(y2)
        
        return x2, y2, {"severity": float(sev)}
    
    def _progress(self, b: ADIBlock, training_step: int) -> float:
        """Compute progress through an ADI block."""
        if b.mode == 'sudden':
            return 1.0
        t = (training_step - b.start) / max(1, b.duration)
        if b.mode == 'gradual':
            ramp = b.transition
            return float(np.clip(t / max(1e-6, ramp), 0.0, 1.0))
        if b.mode == 'incremental':
            return float(np.clip(t, 0.0, 1.0))
        if b.mode == 'recurring':
            phase = (t * 2) % 2.0
            return float(1.0 - abs(phase - 1.0))
        return 1.0
    
    def update_feature_importance(self, x: np.ndarray, gradient_proxy: Optional[np.ndarray] = None):
        """Feature Importance Tracking via EMA."""
        if gradient_proxy is None:
            gradient_proxy = np.abs(x)
        
        importance = np.abs(gradient_proxy) + 1e-8
        importance /= importance.sum()
        
        self.feature_importance = (
            self.importance_decay * self.feature_importance + 
            (1 - self.importance_decay) * importance
        )
    
    def record_attack_outcome(self, success: bool):
        """Track attack success for auto-balancing."""
        self.attack_history.append(float(success))
        
        if len(self.attack_history) >= 20:
            current_success_rate = np.mean(self.attack_history)
            
            if current_success_rate > 0.6:
                self.difficulty_adjustment *= 0.95
            elif current_success_rate < 0.4:
                self.difficulty_adjustment *= 1.05
            
            self.difficulty_adjustment = np.clip(self.difficulty_adjustment, 0.5, 2.0)
    
    def _simulated_fgsm_shift(self, x: np.ndarray, sev: float) -> np.ndarray:
        """Simulated FGSM using feature importance."""
        importance_normalized = self.feature_importance / (self.feature_importance.max() + 1e-8)
        
        perturbation = (
            sev * importance_normalized * 
            self.rng.randn(self.d) * 
            np.sign(self.rng.randn(self.d))
        )
        
        return x + perturbation
    
    def _prior_shift(self, y: Optional[int], sev: float, target: Optional[int]) -> Optional[int]:
        if y is None:
            return None
        if target is None:
            target = 1 - y
        if self.rng.rand() < sev:
            return int(target)
        return y
    
    def _adaptive_conditional_shift(self, x: np.ndarray, y: Optional[int], sev: float) -> Tuple[np.ndarray, Optional[int]]:
        """Adaptive Conditional Shift targeting decision boundary."""
        if y is None:
            return x, None
        
        top_k = max(1, int(0.2 * self.d))
        important_indices = np.argsort(self.feature_importance)[-top_k:]
        
        important_features = x[important_indices]
        feature_range = important_features.max() - important_features.min() + 1e-8
        uncertainty_score = 1.0 - (np.abs(important_features - np.median(important_features)) / feature_range).mean()
        
        flip_prob = sev * uncertainty_score
        
        if self.rng.rand() < flip_prob:
            return x, 1 - y
        
        return x, y
    
    def _poison_label(self, y: Optional[int]) -> Optional[int]:
        if y is None:
            return None
        return 1 - y
    
    @staticmethod
    def _ortho_basis(d: int) -> np.ndarray:
        """Generate orthonormal basis for feature space."""
        if d < 1:
            raise ValueError("Feature dimension must be at least 1")
        
        if d == 1:
            return np.array([[1.0], [1.0]], dtype=np.float32)
        
        basis_count = max(2, min(d, 5))
        A = np.random.RandomState(42).randn(basis_count, d)
        Q, _ = np.linalg.qr(A.T)
        return Q.T.astype(np.float32)


class AdaptiveEnsembleSelector:
    """Adaptive ensemble model selection based on drift severity."""
    
    def __init__(self, model_pool: Dict[str, nn.Module]):
        self.model_pool = model_pool
        self.all_models = list(model_pool.keys())
        self.fast_models = ['mlp_model', 'cnn_model']
        self.medium_models = ['mlp_model', 'cnn_model', 'tcn_model']
        self.all_models_list = ['mlp_model', 'cnn_model', 'tcn_model', 'bilstm_model']
    
    def select_models(self, drift_magnitude: float) -> List[str]:
        if drift_magnitude < 0.3:
            return self.fast_models
        elif drift_magnitude < 0.6:
            return self.medium_models
        else:
            return self.all_models_list
    
    def get_ensemble_subset(self, anchor_idx: int, drift_magnitude: float) -> List[str]:
        """Get ensemble subset with anchor model + adaptive selection."""
        selected = self.select_models(drift_magnitude)
        anchor_name = self.all_models_list[anchor_idx] if anchor_idx < len(self.all_models_list) else self.all_models_list[0]
        
        if anchor_name not in selected:
            selected = [anchor_name] + selected[:-1]
        
        return selected


class DelayedContinualLearning:
    """Continual learning that only activates after N steps."""
    
    def __init__(self, base_continual_defense, enable_after_step: int = 10_000):
        self.base = base_continual_defense
        self.enable_after_step = enable_after_step
        self.current_step = 0
    
    def on_drift_detected(self, current_step: int, drift_vec: np.ndarray):
        self.current_step = current_step
        if current_step >= self.enable_after_step:
            self.base.on_drift_detected(current_step, drift_vec)
    
    def rehearse_old_knowledge(self, *args, **kwargs):
        if self.current_step >= self.enable_after_step:
            return self.base.rehearse_old_knowledge(*args, **kwargs)
        else:
            return 0.0
    
    @property
    def old_knowledge_buffer(self):
        return self.base.old_knowledge_buffer


class BinnedSelfPacedBuffer:
    """Fast self-paced curriculum buffer using difficulty bins."""
    
    def __init__(self, capacity: int = 10_000, n_bins: int = 5):
        self.n_bins = n_bins
        self.bins = [deque(maxlen=capacity // n_bins) for _ in range(n_bins)]
        self.bin_sizes = [0] * n_bins
    
    def push(self, x: np.ndarray, y: int, model_predictions: np.ndarray, loss: float):
        bin_idx = min(int(loss * self.n_bins / 3.0), self.n_bins - 1)
        self.bins[bin_idx].append((x, y))
        self.bin_sizes[bin_idx] = len(self.bins[bin_idx])
    
    def sample_curriculum(self, batch_size: int, curriculum_pace: float) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        max_bin = int(curriculum_pace * self.n_bins)
        target_bins = list(range(max(0, max_bin - 1), self.n_bins))
        
        total_available = sum(len(self.bins[i]) for i in target_bins)
        if total_available == 0:
            return None, None
        
        X_batch, y_batch = [], []
        samples_per_bin = max(1, batch_size // len(target_bins))
        
        for bin_idx in target_bins:
            if len(self.bins[bin_idx]) == 0:
                continue
            
            n_samples = min(samples_per_bin, len(self.bins[bin_idx]))
            indices = np.random.choice(len(self.bins[bin_idx]), n_samples, replace=False)
            
            for idx in indices:
                x, y = self.bins[bin_idx][idx]
                X_batch.append(x)
                y_batch.append(y)
                
                if len(X_batch) >= batch_size:
                    break
            
            if len(X_batch) >= batch_size:
                break
        
        if len(X_batch) == 0:
            return None, None
        
        return np.array(X_batch), np.array(y_batch)
    
    def __len__(self):
        return sum(self.bin_sizes)


# =============================================================================
# REAL COMPARISON METHODS FOR Q1 JOURNAL
# =============================================================================

class IncrementalSGDClassifier:
    """
    Incremental Learning baseline using SGD with partial_fit.
    Standard online learning approach.
    """
    
    def __init__(self, n_classes: int = 2, learning_rate: float = 0.01):
        from sklearn.linear_model import SGDClassifier
        self.model = SGDClassifier(
            loss='log_loss', 
            learning_rate='constant',
            eta0=learning_rate,
            random_state=42,
            warm_start=True
        )
        self.n_classes = n_classes
        self.is_fitted = False
    
    def partial_fit(self, X: np.ndarray, y: np.ndarray):
        """Incremental update."""
        self.model.partial_fit(X, y, classes=np.arange(self.n_classes))
        self.is_fitted = True
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        if not self.is_fitted:
            return np.zeros(len(X), dtype=np.int64)
        return self.model.predict(X)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if not self.is_fitted:
            return np.ones((len(X), self.n_classes)) / self.n_classes
        return self.model.predict_proba(X)


class EnsembleVotingClassifier:
    """
    Static ensemble voting baseline (no RL, just majority vote).
    Compare with RL-guided selection.
    GPU-optimized with AMP support.
    """
    
    def __init__(self, models: Dict[str, nn.Module], device: torch.device):
        self.models = models
        self.device = device
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        all_preds = []
        
        for model in self.models.values():
            model.eval()
            with torch.no_grad():
                if USE_AMP and self.device.type == 'cuda':
                    with autocast():
                        logits = model(x_tensor)
                else:
                    logits = model(x_tensor)
                preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
        
        # Majority voting
        all_preds = np.array(all_preds)
        return np.apply_along_axis(
            lambda x: np.bincount(x, minlength=2).argmax(), 
            axis=0, 
            arr=all_preds
        )
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        all_probs = []
        
        for model in self.models.values():
            model.eval()
            with torch.no_grad():
                if USE_AMP and self.device.type == 'cuda':
                    with autocast():
                        logits = model(x_tensor)
                        probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
                else:
                    logits = model(x_tensor)
                    probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        
        # Average probabilities
        return np.mean(all_probs, axis=0)


class DriftAwareRetrainingClassifier:
    """
    Drift-aware classifier with detector + buffer + retrain mechanism.
    Reference implementation for DDM/ADWIN + Retrain papers.
    """
    
    def __init__(
        self, 
        base_model: nn.Module, 
        detector_type: str = "DDM",
        buffer_size: int = 1000,
        retrain_epochs: int = 5,
        device: torch.device = None,
        retrain_cooldown: int = 500  # Min samples between retrains
    ):
        self.base_model = base_model
        self.device = device or torch.device("cpu")
        self.detector_type = detector_type
        self.buffer_size = buffer_size
        self.retrain_epochs = retrain_epochs
        self.retrain_cooldown = retrain_cooldown
        
        # Initialize detector
        if detector_type == "DDM":
            self.detector = DDM()
        elif detector_type == "ADWIN":
            self.detector = ADWIN()
        elif detector_type == "PageHinkley":
            self.detector = PageHinkley()
        elif detector_type == "KSWIN":
            self.detector = KSWIN()
        else:
            self.detector = DDM()
        
        # Buffer for retraining
        self.X_buffer = deque(maxlen=buffer_size)
        self.y_buffer = deque(maxlen=buffer_size)
        
        # Stats
        self.drift_count = 0
        self.retrain_count = 0
        self.drift_points = []
        self.last_retrain_sample = -retrain_cooldown  # Allow first retrain immediately
        self.sample_count = 0
    
    def update_buffer(self, X: np.ndarray, y: np.ndarray):
        """Add samples to buffer."""
        for i in range(len(X)):
            self.X_buffer.append(X[i])
            self.y_buffer.append(y[i])
            self.sample_count += 1
    
    def check_drift(self, error: float, sample_idx: int = 0) -> bool:
        """Check for drift and trigger retrain if needed."""
        result = self.detector.update(error)
        
        if result["drift"]:
            self.drift_count += 1
            self.drift_points.append(sample_idx)
            
            # Only retrain if cooldown has passed
            if self.sample_count - self.last_retrain_sample >= self.retrain_cooldown:
                self._retrain()
                self.last_retrain_sample = self.sample_count
            
            self.detector.reset()
            return True
        return False
    
    def _retrain(self):
        """Retrain model on buffer data."""
        if len(self.X_buffer) < 100:
            return
        
        X = np.array(list(self.X_buffer))
        y = np.array(list(self.y_buffer))
        
        # Log retraining
        logger.info(f"    → Retraining triggered! Buffer size: {len(X)}, Drift count: {self.drift_count}")
        
        # Simple retraining loop
        self.base_model.train()
        optimizer = torch.optim.Adam(self.base_model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        y_tensor = torch.tensor(y, dtype=torch.long, device=self.device)
        
        for epoch in range(self.retrain_epochs):
            optimizer.zero_grad()
            if USE_AMP and self.device.type == 'cuda':
                with autocast():
                    logits = self.base_model(X_tensor)
                    loss = criterion(logits.float(), y_tensor)
            else:
                logits = self.base_model(X_tensor)
                loss = criterion(logits, y_tensor)
            loss.backward()
            optimizer.step()
        
        self.base_model.eval()
        self.retrain_count += 1
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        self.base_model.eval()
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if USE_AMP and self.device.type == 'cuda':
                with autocast():
                    logits = self.base_model(x_tensor)
            else:
                logits = self.base_model(x_tensor)
            preds = logits.argmax(dim=1).cpu().numpy()
        return preds
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.base_model.eval()
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if USE_AMP and self.device.type == 'cuda':
                with autocast():
                    logits = self.base_model(x_tensor)
                    probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
            else:
                logits = self.base_model(x_tensor)
                probs = F.softmax(logits, dim=-1).cpu().numpy()
        return probs


class SlidingWindowClassifier:
    """
    Sliding window retraining baseline.
    Always retrain on most recent window (no explicit drift detection).
    Reference: Standard online learning approach.
    GPU-optimized with AMP support.
    """
    
    def __init__(
        self,
        base_model: nn.Module,
        window_size: int = 2000,
        retrain_frequency: int = 500,
        retrain_epochs: int = 3,
        device: torch.device = None
    ):
        self.base_model = base_model
        self.device = device or torch.device("cpu")
        self.window_size = window_size
        self.retrain_frequency = retrain_frequency
        self.retrain_epochs = retrain_epochs
        
        self.X_window = deque(maxlen=window_size)
        self.y_window = deque(maxlen=window_size)
        self.sample_count = 0
        self.retrain_count = 0
    
    def update(self, X: np.ndarray, y: np.ndarray):
        """Add samples and retrain if needed."""
        for i in range(len(X)):
            self.X_window.append(X[i])
            self.y_window.append(y[i])
            self.sample_count += 1
        
        # Periodic retraining
        if self.sample_count % self.retrain_frequency == 0 and len(self.X_window) >= 100:
            self._retrain()
    
    def _retrain(self):
        """Retrain on window."""
        X = np.array(list(self.X_window))
        y = np.array(list(self.y_window))
        
        self.base_model.train()
        optimizer = torch.optim.Adam(self.base_model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        y_tensor = torch.tensor(y, dtype=torch.long, device=self.device)
        
        for epoch in range(self.retrain_epochs):
            optimizer.zero_grad()
            logits = self.base_model(X_tensor)
            loss = criterion(logits, y_tensor)
            loss.backward()
            optimizer.step()
        
        self.base_model.eval()
        self.retrain_count += 1
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        self.base_model.eval()
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if USE_AMP and self.device.type == 'cuda':
                with autocast():
                    logits = self.base_model(x_tensor)
            else:
                logits = self.base_model(x_tensor)
            preds = logits.argmax(dim=1).cpu().numpy()
        return preds
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.base_model.eval()
        x_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if USE_AMP and self.device.type == 'cuda':
                with autocast():
                    logits = self.base_model(x_tensor)
                    probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
            else:
                logits = self.base_model(x_tensor)
                probs = F.softmax(logits, dim=-1).cpu().numpy()
        return probs


# =============================================================================
# ENHANCED RL SELECT WITH ALL SOTA COMPONENTS INTEGRATED
# =============================================================================

class EnhancedRLSelectEnsemble(nn.Module):
    """
    Enhanced RL Ensemble with Drift-Aware Selection + ADI + Majority Voting + Incremental Learning:
    
    TWO-TIER DRIFT STRATEGY:
    1. LOW DRIFT (< 0.10): Normal RL model selection
    2. MEDIUM DRIFT (0.10-0.18): Weighted voting + mark for incremental update  
    3. HIGH DRIFT (> 0.18): MAJORITY VOTING across all models for robustness
    
    Components:
    1. SOTADriftEmbedding (20D) - Drift detection AND input to RL state
    2. ADIScheduler - Adversarial Drift Injection with feature importance tracking
    3. Drift Adapter Network - Maps 122D → 102D for pre-trained Q-network
    4. Majority Voting - When drift is high, use consensus of all models
    5. Incremental SGD - Update ALL models when drift detected
    6. Recent Buffer - Store samples for incremental learning
    """
    
    def __init__(self, models: Dict[str, nn.Module], device: torch.device,
                 q_network: nn.Module = None, input_dim: int = 78,
                 selection_temp: float = 1.0, seed: int = 42,
                 incremental_lr: float = 0.001,
                 incremental_steps: int = 10):
        super().__init__()
        self.models = nn.ModuleDict(models)
        self.device = device
        self._q_network = q_network
        self.selection_temp = selection_temp
        self.stochastic_selection = True
        self.last_action = 0
        self.last_def_action, self.last_att_action = 0, 0
        self.last_rewards = (0.0, 0.0)
        self.selection_history = []
        self.rng = np.random.RandomState(seed)
        self.input_dim = input_dim
        
        # Incremental learning params
        self.incremental_lr = incremental_lr
        self.incremental_steps = incremental_steps
        
        # Component 1: SOTA Drift Embedding (20D) - NOW used in RL state
        self.drift_embedding = SOTADriftEmbedding(n_features=input_dim)
        self.lazy_drift = LazySOTADriftEmbedding(self.drift_embedding, update_freq=5)
        self.drift_dim = 20  # SOTADriftEmbedding outputs 20D
        
        # Component 2: ADI Scheduler - Feature importance tracking & adaptive difficulty
        self.adi_scheduler = ADIScheduler(d_features=input_dim, rng=self.rng)
        self.adi_enabled = True  # Can be disabled for ablation
        
        # Component 3: Drift Adapter - Maps enhanced state (102+20=122) to original (102)
        # This allows using pre-trained Q-network with drift-aware input
        self.drift_adapter = nn.Sequential(
            nn.Linear(102 + self.drift_dim, 128),  # 122 → 128
            nn.ReLU(),
            nn.Linear(128, 102)  # 128 → 102 (original Q-network input)
        ).to(device)
        
        # Initialize adapter with identity-like mapping for base features
        with torch.no_grad():
            # First layer: pass through base features, compress drift
            self.drift_adapter[0].weight[:102, :102] = torch.eye(102) * 0.5
            self.drift_adapter[0].bias.zero_()
            # Second layer: reconstruct
            self.drift_adapter[2].weight.normal_(0, 0.1)
            self.drift_adapter[2].bias.zero_()
        
        # Component 4: Buffer for incremental learning
        self.recent_buffer_X = deque(maxlen=500)
        self.recent_buffer_y = deque(maxlen=500)
        
        # Drift tracking
        self.current_drift_magnitude = 0.0
        self.drift_history = deque(maxlen=100)
        
        # Drift detection thresholds (TWO-TIER) - TUNED for better adaptation
        self.drift_threshold_low = 0.10    # TUNED: Lower threshold for earlier detection
        self.drift_threshold_high = 0.18   # TUNED: Lower high threshold for more voting when needed
        self.drift_detection_count = 0
        self.drift_points = []
        self.sample_counter = 0
        self.last_incremental_update = -1000  # Cooldown tracking
        self.incremental_cooldown = 20  # TUNED: Much more frequent updates (was 50)
        self._pending_incremental_model = None  # Model idx pending incremental update
        
        # Stats
        self.adaptive_selection_counts = {name: 0 for name in models.keys()}
        self.drift_triggered_adaptations = 0
        self.incremental_updates = 0
        self.majority_voting_count = 0  # Track when voting is used
        
        # For plotting: Track tau (temperature) and drift over time
        self.tau_history = []  # Adaptive temperature history
        self.drift_magnitude_history = []  # Drift magnitude for each forward pass
        
        # Catastrophic forgetting tracking
        self.pre_drift_buffer_X = deque(maxlen=200)  # Store samples from before drift
        self.pre_drift_buffer_y = deque(maxlen=200)
        self.forgetting_scores = []  # Track performance on old data after updates
    
    @property
    def model_list(self):
        return list(self.models.values())
    
    @property
    def model_names(self):
        return list(self.models.keys())
    
    def push_reference(self, x: np.ndarray, score: float = None, y: int = None):
        """Push sample to reference distribution (for drift baseline)."""
        self.drift_embedding.push(x, score, y, to_ref=True)
    
    def push_sample(self, x: np.ndarray, score: float = None, y: int = None):
        """Push sample to recent distribution (for drift detection)."""
        self.lazy_drift.push(x, score, y, to_ref=False)
    
    def push_to_buffer(self, x: np.ndarray, y: int):
        """Push sample to recent buffer for incremental learning."""
        self.recent_buffer_X.append(x)
        self.recent_buffer_y.append(y)
    
    def _compute_drift_magnitude(self) -> float:
        """
        Compute drift magnitude combining:
        1. SOTA Drift Embedding (distribution shift)
        2. ADI feature importance (adversarial drift indicators)
        """
        drift_vec = self.lazy_drift.vector()
        base_magnitude = float(np.mean(drift_vec[:4]))  # Distribution metrics
        
        # ADI-enhanced drift detection
        if self.adi_enabled and hasattr(self, 'adi_scheduler'):
            # Get ADI's feature importance entropy as drift indicator
            importance = self.adi_scheduler.feature_importance
            importance_entropy = -np.sum(importance * np.log(importance + 1e-8))
            max_entropy = np.log(len(importance))
            normalized_entropy = importance_entropy / max_entropy if max_entropy > 0 else 0
            
            # High entropy = features changing importance = potential drift
            adi_drift_signal = 1.0 - normalized_entropy  # Low entropy = concentrated = drift
            
            # ADI difficulty adjustment as drift proxy
            difficulty_drift = abs(self.adi_scheduler.difficulty_adjustment - 1.0)
            
            # Combine signals: base_magnitude weighted most, ADI signals as boosters
            magnitude = base_magnitude * 0.7 + adi_drift_signal * 0.2 + difficulty_drift * 0.1
        else:
            magnitude = base_magnitude
        
        self.current_drift_magnitude = magnitude
        self.drift_history.append(magnitude)
        return magnitude
    
    def update_adi_from_prediction(self, x: np.ndarray, y_true: int, y_pred: int):
        """Update ADI scheduler based on prediction outcome."""
        if not self.adi_enabled:
            return
        
        # Update feature importance using input as gradient proxy
        self.adi_scheduler.update_feature_importance(x)
        
        # Record attack outcome (incorrect prediction = successful "attack")
        attack_success = (y_pred != y_true)
        self.adi_scheduler.record_attack_outcome(attack_success)
    
    def _incremental_update(self, model_idx: int):
        """Perform incremental update on selected model using recent buffer."""
        if len(self.recent_buffer_X) < 50:
            return
        
        # Check cooldown
        if self.sample_counter - self.last_incremental_update < self.incremental_cooldown:
            return
        
        self.last_incremental_update = self.sample_counter
        self.incremental_updates += 1
        
        # Get recent data
        X = np.array(list(self.recent_buffer_X))
        y = np.array(list(self.recent_buffer_y))
        
        # ADI-enhanced: Weight samples by feature importance
        if self.adi_enabled and hasattr(self, 'adi_scheduler'):
            # Compute sample weights based on ADI feature importance
            importance = self.adi_scheduler.feature_importance
            sample_importance = np.abs(X @ importance)  # Project samples onto importance
            sample_weights = sample_importance / (sample_importance.sum() + 1e-8)
            sample_weights = 0.5 + 0.5 * sample_weights  # Normalize to [0.5, 1.0] range
            sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.float32, device=self.device)
        else:
            sample_weights = None
            sample_weights_tensor = None
        
        # TUNED: Weight recent samples more with class-balanced loss
        # Compute class weights for imbalanced data handling
        unique_classes, counts = np.unique(y, return_counts=True)
        if len(unique_classes) > 1:
            total = sum(counts)
            class_weights = torch.tensor([total / (len(unique_classes) * c) for c in counts],
                                        dtype=torch.float32, device=self.device)
        else:
            class_weights = None
        
        # Update ALL models when drift detected (not just selected one)
        for idx, model in enumerate(self.model_list):
            model.train()
            
            # TUNED: Higher LR for faster adaptation during drift
            # ADI-adjusted LR based on difficulty
            adi_lr_mult = self.adi_scheduler.difficulty_adjustment if self.adi_enabled else 1.0
            optimizer = torch.optim.SGD(model.parameters(), 
                                       lr=self.incremental_lr * 2.0 * adi_lr_mult,  # ADI-adjusted
                                       momentum=0.9,
                                       weight_decay=1e-5)  # Slight regularization
            
            if class_weights is not None:
                criterion = nn.CrossEntropyLoss(weight=class_weights, reduction='none')
            else:
                criterion = nn.CrossEntropyLoss(reduction='none')
            
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            y_tensor = torch.tensor(y, dtype=torch.long, device=self.device)
            
            # TUNED: More incremental steps for better learning
            for _ in range(self.incremental_steps * 2):  # TUNED: 2x steps
                optimizer.zero_grad()
                logits = model(X_tensor)
                per_sample_loss = criterion(logits, y_tensor)
                
                # ADI-weighted loss: focus on important samples
                if sample_weights_tensor is not None:
                    loss = (per_sample_loss * sample_weights_tensor).mean()
                else:
                    loss = per_sample_loss.mean()
                
                loss.backward()
                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            model.eval()
        
        logger.info(f"    → Incremental update on ALL {len(self.model_list)} models, buffer={len(X)}")
        
        # Measure catastrophic forgetting: evaluate on pre-drift data
        if len(self.pre_drift_buffer_X) >= 50:
            self._measure_forgetting()
    
    def _measure_forgetting(self):
        """Measure performance on old (pre-drift) data after incremental update."""
        if len(self.pre_drift_buffer_X) < 50:
            return
        
        X_old = np.array(list(self.pre_drift_buffer_X))
        y_old = np.array(list(self.pre_drift_buffer_y))
        
        X_tensor = torch.tensor(X_old, dtype=torch.float32, device=self.device)
        y_tensor = torch.tensor(y_old, dtype=torch.long, device=self.device)
        
        # Evaluate each model on old data
        total_correct = 0
        total_samples = len(y_old)
        
        for model in self.model_list:
            model.eval()
            with torch.no_grad():
                logits = model(X_tensor)
                preds = logits.argmax(dim=-1)
                correct = (preds == y_tensor).sum().item()
                total_correct += correct
        
        # Average accuracy across all models on old data
        avg_acc_on_old = total_correct / (total_samples * len(self.model_list))
        self.forgetting_scores.append({
            'sample_idx': self.sample_counter,
            'acc_on_old_data': avg_acc_on_old,
            'n_updates': self.incremental_updates
        })
    
    def update_pre_drift_buffer(self, x: np.ndarray, y: int):
        """Store samples as 'pre-drift' baseline for forgetting measurement."""
        # Only add if drift is low (this is 'old' distribution data)
        if self.current_drift_magnitude < self.drift_threshold_low:
            self.pre_drift_buffer_X.append(x)
            self.pre_drift_buffer_y.append(y)
    
    def trigger_pending_incremental_update(self):
        """Execute pending incremental update (call OUTSIDE of no_grad context)."""
        if self._pending_incremental_model is not None:
            self._incremental_update(self._pending_incremental_model)
            self._pending_incremental_model = None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RL model selection with DRIFT EMBEDDING in state + Incremental learning.
        
        STRATEGY:
        1. Compute drift embedding (20D vector)
        2. Build enhanced state [base(102) + drift(20)] = 122D
        3. Use drift_adapter to map 122D → 102D for pre-trained Q-network
        4. If drift detected → trigger incremental update on selected model
        5. Execute selected model
        """
        num_models = len(self.model_list)
        self.sample_counter += 1
        
        # Step 1: Update drift embedding and compute magnitude
        x_np = x.mean(dim=0).cpu().numpy()
        self.push_sample(x_np)
        drift_magnitude = self._compute_drift_magnitude()
        
        # Get full 20D drift vector for RL state
        drift_vec = self.lazy_drift.vector()  # 20D drift embedding
        drift_tensor = torch.tensor(drift_vec, device=x.device, dtype=x.dtype)
        
        # Step 2: RL model selection WITH DRIFT EMBEDDING
        if self._q_network is not None:
            # Build base state (102D): [features(76) + def_oh(12) + att_oh(12) + rewards(2)]
            feat = x.mean(dim=0)
            def_oh = torch.zeros(12, device=x.device, dtype=x.dtype)
            def_oh[self.last_def_action] = 1.0
            att_oh = torch.zeros(12, device=x.device, dtype=x.dtype)
            att_oh[self.last_att_action] = 1.0
            rewards = torch.tensor(self.last_rewards, device=x.device, dtype=x.dtype)
            base_state = torch.cat([feat, def_oh, att_oh, rewards])  # 102D
            
            # ENHANCED STATE: [base(102) + drift(20)] = 122D
            enhanced_state = torch.cat([base_state, drift_tensor]).unsqueeze(0)
            
            # Adapt 122D → 102D for pre-trained Q-network
            adapted_state = self.drift_adapter(enhanced_state)
            
            with torch.no_grad():
                q_vals = self._q_network(adapted_state).squeeze(0)
            
            # TUNED: Adaptive temperature based on drift - more exploration under drift
            base_temp = self.selection_temp
            if drift_magnitude > self.drift_threshold_high:
                # High drift: much more exploration
                adaptive_temp = base_temp * (2.0 + drift_magnitude * 3.0)
            elif drift_magnitude > self.drift_threshold_low:
                # Medium drift: moderate exploration
                adaptive_temp = base_temp * (1.5 + drift_magnitude * 2.0)
            else:
                # Low drift: normal temperature
                adaptive_temp = base_temp * (1.0 + drift_magnitude)
            
            # Track tau and drift for plotting
            self.tau_history.append(adaptive_temp)
            self.drift_magnitude_history.append(drift_magnitude)
            
            if self.stochastic_selection:
                probs = F.softmax(q_vals / adaptive_temp, dim=-1)
                action = torch.multinomial(probs, 1).item()
            else:
                action = q_vals.argmax().item()
            
            self.last_action = action
            self.last_def_action = action
            
            # Action mapping: action space 12 → 4 models
            model_idx = min(action // 3, num_models - 1)
        else:
            # Fallback: Use first model
            model_idx = 0
        
        # Step 3: Check drift level and decide strategy
        use_voting = False
        use_weighted_voting = False  # TUNED: New option for medium drift
        
        if drift_magnitude > self.drift_threshold_high:
            # HIGH DRIFT: Use majority voting for robustness
            self.drift_detection_count += 1
            self.drift_triggered_adaptations += 1
            self.majority_voting_count += 1
            use_voting = True
            
            # Record drift point (with debounce)
            if len(self.drift_points) == 0 or self.sample_counter - self.drift_points[-1] > 50:  # TUNED: shorter debounce
                self.drift_points.append(self.sample_counter)
            
            # Mark for incremental update
            self._pending_incremental_model = model_idx
            
        elif drift_magnitude > self.drift_threshold_low:
            # MEDIUM DRIFT: Use weighted average (not hard voting)
            self.drift_detection_count += 1
            use_weighted_voting = True  # TUNED: Softer combination
            self._pending_incremental_model = model_idx
        
        # Step 4: Execute model(s)
        if use_voting:
            # MAJORITY VOTING: Run all models and vote
            all_preds = []
            all_logits = []
            for model in self.model_list:
                model.eval()
                with torch.no_grad():
                    logits = model(x)
                    preds = logits.argmax(dim=-1)
                    all_preds.append(preds)
                    all_logits.append(logits)
            
            # Stack predictions and do majority vote
            stacked_preds = torch.stack(all_preds, dim=0)  # [n_models, batch]
            # Majority vote per sample
            voted_preds = torch.mode(stacked_preds, dim=0).values
            
            # Average logits for probability output
            logits = torch.stack(all_logits).mean(dim=0)
            
            self.selection_history.append(-1)  # -1 = voting
            for name in self.model_names:
                self.adaptive_selection_counts[name] += 1
                
        elif use_weighted_voting:
            # MEDIUM DRIFT: Weighted average based on Q-values (softer than hard voting)
            all_logits = []
            # Use Q-values for weighting if available
            if self._q_network is not None:
                try:
                    q_weights = F.softmax(q_vals[:num_models] / 0.5, dim=-1)
                except:
                    q_weights = torch.ones(num_models, device=x.device) / num_models
            else:
                q_weights = torch.ones(num_models, device=x.device) / num_models
            
            for idx, model in enumerate(self.model_list):
                model.eval()
                with torch.no_grad():
                    model_logits = model(x)
                    # Weight by Q-value confidence
                    all_logits.append(model_logits * q_weights[idx].item())
            
            # Sum weighted logits
            logits = torch.stack(all_logits).sum(dim=0)
            
            self.selection_history.append(-2)  # -2 = weighted voting
            for name in self.model_names:
                self.adaptive_selection_counts[name] += 0.25  # Partial credit
        else:
            # Normal RL selection
            self.selection_history.append(model_idx)
            model_name = self.model_names[model_idx]
            self.adaptive_selection_counts[model_name] += 1
            
            selected_model = self.model_list[model_idx]
            selected_model.eval()
            with torch.no_grad():
                logits = selected_model(x)
        
        return logits
    
    def get_selection_stats(self) -> Dict[str, int]:
        """Get model selection statistics."""
        stats = {}
        for i, name in enumerate(self.model_names):
            stats[name] = self.selection_history.count(i)
        stats['ensemble_voting'] = self.selection_history.count(-1)
        stats['weighted_voting'] = self.selection_history.count(-2)
        return stats
    
    def get_drift_stats(self) -> Dict[str, float]:
        """Get comprehensive drift statistics."""
        if len(self.drift_history) == 0:
            return {
                "mean_drift": 0.0,
                "max_drift": 0.0,
                "current_drift": 0.0,
                "drift_triggered_adaptations": 0,
                "drift_detection_count": 0,
                "incremental_updates": self.incremental_updates,
                "drift_points": [],
                "adi_difficulty": 1.0,
                "adi_attack_success_rate": 0.0
            }
        
        # ADI stats
        adi_difficulty = self.adi_scheduler.difficulty_adjustment if self.adi_enabled else 1.0
        adi_success_rate = float(np.mean(self.adi_scheduler.attack_history)) if self.adi_scheduler.attack_history else 0.0
        
        return {
            "mean_drift": float(np.mean(self.drift_history)),
            "max_drift": float(np.max(self.drift_history)),
            "min_drift": float(np.min(self.drift_history)),
            "current_drift": self.current_drift_magnitude,
            "drift_std": float(np.std(self.drift_history)),
            "drift_triggered_adaptations": self.drift_triggered_adaptations,
            "drift_detection_count": self.drift_detection_count,
            "incremental_updates": self.incremental_updates,
            "majority_voting_count": self.majority_voting_count,
            "drift_points": self.drift_points[:20],  # First 20
            # ADI stats
            "adi_difficulty": adi_difficulty,
            "adi_attack_success_rate": adi_success_rate,
            "adi_enabled": self.adi_enabled
        }
    
    def get_drift_embedding_interpretation(self) -> Dict[str, float]:
        """Get interpreted drift embedding for analysis."""
        drift_vec = self.drift_embedding.vector()
        return self.drift_embedding.interpret(drift_vec)
    
    def get_buffer_stats(self) -> Dict:
        """Get incremental buffer statistics."""
        return {
            "buffer_size_X": len(self.recent_buffer_X),
            "buffer_size_y": len(self.recent_buffer_y),
            "incremental_updates": self.incremental_updates
        }
    
    def get_all_stats(self) -> Dict:
        """Get all statistics for comprehensive analysis."""
        return {
            "selection_stats": self.get_selection_stats(),
            "drift_stats": self.get_drift_stats(),
            "buffer_stats": self.get_buffer_stats(),
            "adaptive_selection_counts": self.adaptive_selection_counts
        }
    
    def reset_history(self):
        self.selection_history = []
        self.drift_triggered_adaptations = 0
    
    def enable_stochastic(self): 
        self.stochastic_selection = True
    
    def disable_stochastic(self): 
        self.stochastic_selection = False


# =============================================================================
# LEGACY CLASSES (kept for backward compatibility, but main method is EnhancedRLSelectEnsemble)
# =============================================================================
# COMPUTE DRIFT METRICS
# =============================================================================

def compute_recovery_time(accuracies: np.ndarray, baseline_acc: float, drift_point_idx: int, recovery_threshold: float = 0.90) -> int:
    """Compute time to recover to threshold% of baseline accuracy."""
    target_acc = baseline_acc * recovery_threshold
    for i, acc in enumerate(accuracies[drift_point_idx:]):
        if acc >= target_acc:
            return i
    return -1


def compute_all_drift_metrics(accuracies: np.ndarray, baseline_acc: float, drift_point_idx: int, 
                               detection_point: int = -1, config: DriftConfig = None) -> DriftMetrics:
    """Compute all drift adaptation metrics."""
    config = config or DriftConfig()
    
    recovery_time = compute_recovery_time(accuracies, baseline_acc, drift_point_idx, config.recovery_threshold)
    post_drift_accs = accuracies[drift_point_idx:]
    accuracy_drop = baseline_acc - post_drift_accs.min() if len(post_drift_accs) > 0 else 0.0
    
    if recovery_time > 0 and len(post_drift_accs) > 1:
        min_acc_idx = np.argmin(post_drift_accs)
        recovery_accs = post_drift_accs[min_acc_idx:min(min_acc_idx + recovery_time + 1, len(post_drift_accs))]
        recovery_rate = (recovery_accs[-1] - recovery_accs[0]) / max(1, len(recovery_accs) / 1000) if len(recovery_accs) > 1 else 0.0
    else:
        recovery_rate = 0.0
    
    final_accuracy = accuracies[-10:].mean() if len(accuracies) >= 10 else accuracies.mean()
    stability_index = accuracies[drift_point_idx:].var() if len(accuracies) > drift_point_idx else 0.0
    detection_delay = detection_point - drift_point_idx * config.eval_window if detection_point >= 0 else -1
    total_errors = int(np.sum(1 - post_drift_accs) * config.eval_window) if len(post_drift_accs) > 0 else 0
    
    return DriftMetrics(
        detection_delay=detection_delay,
        recovery_time=recovery_time * config.eval_window if recovery_time > 0 else -1,
        recovery_rate=recovery_rate,
        accuracy_drop=accuracy_drop,
        final_accuracy=final_accuracy,
        prequential_accuracy=accuracies.mean(),
        stability_index=stability_index,
        total_errors_during_drift=total_errors
    )


# =============================================================================
# MAIN DRIFT EVALUATION WITH REAL MODELS - Q1 JOURNAL VERSION
# =============================================================================

def load_full_data_for_drift(data_dir: str, scaler_path: str, num_samples: int = 10000, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """Load larger dataset for drift evaluation (both benign and attack) - Synthetic Drift."""
    csv_files = [
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
        "Tuesday-WorkingHours.pcap_ISCX.csv",
        "Monday-WorkingHours.pcap_ISCX.csv",  # Mostly benign
    ]
    drop_cols = ["Flow ID", "Fwd Header Length.1", "Source IP", "Src IP", "Source Port", "Src Port",
                 "Destination IP", "Dst IP", "Destination Port", "Dst Port", "Timestamp"]
    
    dfs = []
    for f in csv_files:
        path = os.path.join(data_dir, f)
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            df.drop(columns=drop_cols, inplace=True, errors="ignore")
            df["Label"] = df["Label"].replace({"BENIGN": "Benign"})
            dfs.append(df)
    
    if not dfs:
        logger.warning("No data files found, generating synthetic data")
        np.random.seed(seed)
        X = np.random.randn(num_samples, 78).astype(np.float32)
        y = (np.random.rand(num_samples) > 0.7).astype(np.int64)
        return X, y
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_all.dropna(inplace=True)
    
    np.random.seed(seed)
    if len(df_all) > num_samples:
        df_all = df_all.sample(n=num_samples, random_state=seed)
    
    X = df_all.drop("Label", axis=1).values.astype(np.float32)
    y = (df_all["Label"] != "Benign").astype(np.int64).values
    
    if os.path.exists(scaler_path):
        X = joblib.load(scaler_path).transform(X).astype(np.float32)
    
    logger.info(f"Loaded {len(X)} samples for drift eval, {X.shape[1]} features, Attack ratio: {y.mean():.2%}")
    return X, y


def evaluate_temporal_drift(
    model_pool: Dict[str, nn.Module],
    q_network: nn.Module,
    data_dict: Dict,
    device: torch.device,
    eval_window: int = 200
) -> Dict:
    """
    Evaluate drift adaptation on REAL temporal drift (CIC-IDS2017 days).
    
    This is the MAIN evaluation for Q1 journal - uses actual temporal data
    where different days have different attack distributions.
    
    Methods compared (baselines):
    1. Static_Best: Best single model from pool (no adaptation)
    2. Ensemble_Voting: Static ensemble with majority voting
    3. DDM_Retrain: DDM drift detection + retrain
    4. ADWIN_Retrain: ADWIN drift detection + retrain
    5. Sliding_Window: Periodic retraining on sliding window
    6. Incremental_SGD: Online learning with SGD
    
    Our Method (Enhanced RL):
    7. Ours_RLSelect: Enhanced RL-guided dynamic model selection
       Integrates ALL SOTA components:
       - SOTADriftEmbedding (20D drift vector)
       - LazySOTADriftEmbedding (cached computation)
       - AdaptiveEnsembleSelector (drift-based model subset)
       - BinnedSelfPacedBuffer (curriculum learning)
    """
    results = {}
    
    X_train, y_train, train_attacks = data_dict["train"]
    test_days = data_dict["test_days"]
    
    if len(X_train) == 0:
        logger.error("No training data available")
        return results
    
    input_dim = X_train.shape[1]
    
    logger.info(f"\n{'='*60}")
    logger.info(f"TEMPORAL DRIFT EVALUATION - Q1 Journal")
    logger.info(f"Training on: {train_attacks} ({len(X_train)} samples)")
    logger.info(f"Testing on {len(test_days)} days")
    logger.info(f"{'='*60}")
    
    # ==========================================================================
    # Prepare all methods
    # ==========================================================================
    
    methods = {}
    
    # 1. Static Best Model
    best_model_name = list(model_pool.keys())[0]
    best_model = model_pool[best_model_name]
    methods["Static_Best"] = {
        "type": "static",
        "model": best_model,
        "name": f"Static ({best_model_name})"
    }
    
    # 2. Ensemble Voting
    methods["Ensemble_Voting"] = {
        "type": "ensemble_vote",
        "model": EnsembleVotingClassifier(model_pool, device),
        "name": "Ensemble (Voting)"
    }
    
    # 3. DDM + Retrain
    ddm_model = copy.deepcopy(best_model)
    methods["DDM_Retrain"] = {
        "type": "drift_aware",
        "model": DriftAwareRetrainingClassifier(ddm_model, "DDM", device=device),
        "name": "DDM + Retrain"
    }
    
    # 4. ADWIN + Retrain
    adwin_model = copy.deepcopy(best_model)
    methods["ADWIN_Retrain"] = {
        "type": "drift_aware",
        "model": DriftAwareRetrainingClassifier(adwin_model, "ADWIN", device=device),
        "name": "ADWIN + Retrain"
    }
    
    # 5. Sliding Window Retrain
    sw_model = copy.deepcopy(best_model)
    methods["Sliding_Window"] = {
        "type": "sliding_window",
        "model": SlidingWindowClassifier(sw_model, window_size=2000, device=device),
        "name": "Sliding Window"
    }
    
    # 6. Incremental SGD
    methods["Incremental_SGD"] = {
        "type": "incremental",
        "model": IncrementalSGDClassifier(n_classes=2),
        "name": "Incremental SGD"
    }
    
    # 7. Our Enhanced RL Select Ensemble (MAIN METHOD - integrates ALL components)
    # This is the MAIN "Ours" method that combines:
    # - SOTADriftEmbedding (20D drift state) for drift DETECTION only
    # - Incremental SGD learner for quick adaptation
    # - Recent buffer for incremental learning samples
    enhanced_rl_ensemble = EnhancedRLSelectEnsemble(
        models=model_pool,
        device=device,
        q_network=q_network,
        input_dim=input_dim,
        selection_temp=1.0,
        seed=42,
        incremental_lr=0.001,
        incremental_steps=10
    ).to(device)
    enhanced_rl_ensemble.eval()
    enhanced_rl_ensemble.enable_stochastic()
    methods["Ours_RLSelect"] = {
        "type": "enhanced_rl_select",
        "model": enhanced_rl_ensemble,
        "name": "Ours (Enhanced RL)"
    }
    
    # ==========================================================================
    # Initialize methods with training data
    # ==========================================================================
    
    logger.info("\nInitializing methods with training data...")
    
    # Train incremental classifier
    methods["Incremental_SGD"]["model"].partial_fit(X_train, y_train)
    
    # Fill buffers for drift-aware methods
    methods["DDM_Retrain"]["model"].update_buffer(X_train[-1000:], y_train[-1000:])
    methods["ADWIN_Retrain"]["model"].update_buffer(X_train[-1000:], y_train[-1000:])
    methods["Sliding_Window"]["model"].update(X_train[-2000:], y_train[-2000:])
    
    # Initialize Enhanced RL with reference distribution (for drift detection baseline)
    # Push training samples to drift embedding as reference
    logger.info("Initializing Enhanced RL reference distribution...")
    for i in range(min(1000, len(X_train))):
        methods["Ours_RLSelect"]["model"].push_reference(X_train[i], y=y_train[i])
    
    # Pre-fill incremental buffer with some training data
    for i in range(min(500, len(X_train))):
        methods["Ours_RLSelect"]["model"].push_to_buffer(X_train[-500+i], y_train[-500+i])
    
    # ==========================================================================
    # Evaluate on each test day (temporal progression)
    # ==========================================================================
    
    for method_name in methods:
        results[method_name] = {
            "per_day": {},
            "accuracies_over_time": [],
            "f1_over_time": [],
            "dr_over_time": [],
            "auc_over_time": [],  # Add AUC tracking
            "precision_over_time": [],
            "day_boundaries": [],
            "drift_detections": [],
            "all_preds": [],  # For overall AUC computation
            "all_probs": [],
            "all_labels": []
        }
    
    global_sample_idx = 0
    
    for day_idx, (day_name, X_day, y_day, attacks) in enumerate(test_days):
        logger.info(f"\n--- Day: {day_name} ({len(X_day)} samples, Attacks: {attacks}) ---")
        
        n_windows = max(1, len(X_day) // eval_window)
        
        # Store day boundary for ALL methods (not just first)
        for method_name in methods:
            results[method_name]["day_boundaries"].append(global_sample_idx)
        
        for method_name, method_info in methods.items():
            day_preds = []
            day_labels = []
            day_probs = []  # Collect probabilities for AUC
            window_accs = []
            window_f1s = []
            window_drs = []
            window_aucs = []
            window_precisions = []
            
            for w_idx in range(n_windows):
                start_idx = w_idx * eval_window
                end_idx = min(start_idx + eval_window, len(X_day))
                
                X_window = X_day[start_idx:end_idx]
                y_window = y_day[start_idx:end_idx]
                
                # Get predictions AND probabilities based on method type
                probs = None  # Initialize probs
                
                if method_info["type"] == "static":
                    model = method_info["model"]
                    # Use GPU-optimized batch inference
                    preds, probs = gpu_batch_inference(model, X_window, device)
                
                elif method_info["type"] == "ensemble_vote":
                    model = method_info["model"]
                    preds = model.predict(X_window)
                    probs = model.predict_proba(X_window)
                
                elif method_info["type"] == "drift_aware":
                    model = method_info["model"]
                    preds = model.predict(X_window)
                    probs = model.predict_proba(X_window)
                    # Update buffer and check drift PER SAMPLE (not per window)
                    model.update_buffer(X_window, y_window)
                    # Feed each sample's error to detector for proper sensitivity
                    for i, (pred, label) in enumerate(zip(preds, y_window)):
                        sample_error = float(pred != label)  # 0 or 1
                        sample_idx = global_sample_idx + start_idx + i
                        if model.check_drift(sample_error, sample_idx):
                            results[method_name]["drift_detections"].append(sample_idx)
                            break  # Only record first drift in window
                
                elif method_info["type"] == "sliding_window":
                    model = method_info["model"]
                    preds = model.predict(X_window)
                    probs = model.predict_proba(X_window)
                    model.update(X_window, y_window)
                
                elif method_info["type"] == "incremental":
                    model = method_info["model"]
                    preds = model.predict(X_window)
                    probs = model.predict_proba(X_window)
                    model.partial_fit(X_window, y_window)
                
                elif method_info["type"] == "rl_select":
                    model = method_info["model"]
                    # Use GPU-optimized batch inference with AMP
                    preds, probs = gpu_batch_inference(model, X_window, device)
                
                elif method_info["type"] == "enhanced_rl_select":
                    # MAIN METHOD: Enhanced RL with ADI + Incremental Learning
                    model = method_info["model"]
                    # Use GPU-optimized batch inference with AMP
                    preds, probs = gpu_batch_inference(model, X_window, device)
                    
                    # Update drift tracker, ADI, and incremental buffer with current samples
                    for i in range(len(X_window)):
                        model.push_sample(X_window[i], y=y_window[i])
                        model.push_to_buffer(X_window[i], y_window[i])
                        model.update_pre_drift_buffer(X_window[i], y_window[i])  # For forgetting metric
                        
                        # ADI feedback: update feature importance based on prediction outcome
                        model.update_adi_from_prediction(X_window[i], y_window[i], preds[i])
                    
                    # Trigger pending incremental update (OUTSIDE no_grad context)
                    model.trigger_pending_incremental_update()
                    
                    # Update drift detections from EnhancedRL model
                    results[method_name]["drift_detections"] = model.drift_points.copy()
                    
                    # Store tau and drift history for plotting
                    results[method_name]["tau_history"] = model.tau_history.copy()
                    results[method_name]["drift_magnitude_history"] = model.drift_magnitude_history.copy()
                    results[method_name]["forgetting_scores"] = model.forgetting_scores.copy()
                    
                    # Store ADI stats
                    results[method_name]["adi_difficulty"] = model.adi_scheduler.difficulty_adjustment
                    results[method_name]["adi_success_rate"] = np.mean(model.adi_scheduler.attack_history) if model.adi_scheduler.attack_history else 0.0
                
                # Compute window metrics
                acc = accuracy_score(y_window, preds)
                f1 = f1_score(y_window, preds, average='binary', zero_division=0)
                dr = recall_score(y_window, preds, average='binary', zero_division=0)
                prec = precision_score(y_window, preds, average='binary', zero_division=0)
                
                # Compute AUC-ROC (requires both classes present)
                try:
                    if len(np.unique(y_window)) >= 2 and probs is not None:
                        auc = roc_auc_score(y_window, probs[:, 1])
                    else:
                        auc = np.nan
                except (ValueError, IndexError):
                    auc = np.nan
                
                window_accs.append(acc)
                window_f1s.append(f1)
                window_drs.append(dr)
                window_aucs.append(auc)
                window_precisions.append(prec)
                
                # Store for overall metrics
                day_preds.extend(preds.tolist())
                day_labels.extend(y_window.tolist())
                if probs is not None:
                    day_probs.extend(probs[:, 1].tolist())  # Probability of positive class
            
            # Store window-level metrics
            results[method_name]["accuracies_over_time"].extend(window_accs)
            results[method_name]["f1_over_time"].extend(window_f1s)
            results[method_name]["dr_over_time"].extend(window_drs)
            results[method_name]["auc_over_time"].extend(window_aucs)
            results[method_name]["precision_over_time"].extend(window_precisions)
            
            # Store all predictions/probs for overall AUC
            results[method_name]["all_preds"].extend(day_preds)
            results[method_name]["all_labels"].extend(day_labels)
            results[method_name]["all_probs"].extend(day_probs)
            
            # Convert to numpy arrays for computation
            day_preds_arr = np.array(day_preds)
            day_labels_arr = np.array(day_labels)
            day_probs_arr = np.array(day_probs)
            
            # Compute day-level AUC
            try:
                if len(np.unique(day_labels_arr)) >= 2 and len(day_probs_arr) == len(day_labels_arr) and len(day_probs_arr) > 0:
                    day_auc = float(roc_auc_score(day_labels_arr, day_probs_arr))
                else:
                    day_auc = np.nan
            except (ValueError, IndexError):
                day_auc = np.nan
            
            day_metrics = {
                "accuracy": float(accuracy_score(day_labels_arr, day_preds_arr)),
                "f1": float(f1_score(day_labels_arr, day_preds_arr, average='binary', zero_division=0)),
                "detection_rate": float(recall_score(day_labels_arr, day_preds_arr, average='binary', zero_division=0)),
                "precision": float(precision_score(day_labels_arr, day_preds_arr, average='binary', zero_division=0)),
                "auc_roc": day_auc,
                "n_samples": len(day_labels_arr),
                "attack_ratio": float(np.mean(day_labels_arr)) if len(day_labels_arr) > 0 else 0.0,
                "attacks": attacks
            }
            results[method_name]["per_day"][day_name] = day_metrics
            
            auc_str = f"{day_auc:.4f}" if not np.isnan(day_auc) else "N/A"
            logger.info(f"  {method_info['name']}: Acc={day_metrics['accuracy']:.4f}, "
                       f"DR={day_metrics['detection_rate']:.4f}, F1={day_metrics['f1']:.4f}, AUC={auc_str}")
        
        global_sample_idx += len(X_day)
    
    # ==========================================================================
    # Compute summary metrics
    # ==========================================================================
    
    logger.info(f"\n{'='*60}")
    logger.info("SUMMARY METRICS")
    logger.info(f"{'='*60}")
    
    for method_name in methods:
        accs = np.array(results[method_name]["accuracies_over_time"])
        f1s = np.array(results[method_name]["f1_over_time"])
        drs = np.array(results[method_name]["dr_over_time"])
        aucs = np.array(results[method_name]["auc_over_time"])
        precs = np.array(results[method_name]["precision_over_time"])
        
        # Compute overall AUC from all predictions
        all_labels = np.array(results[method_name]["all_labels"])
        all_probs = np.array(results[method_name]["all_probs"])
        try:
            if len(np.unique(all_labels)) >= 2 and len(all_probs) == len(all_labels):
                overall_auc = float(roc_auc_score(all_labels, all_probs))
            else:
                overall_auc = np.nan
        except (ValueError, IndexError):
            overall_auc = np.nan
        
        # Mean AUC (excluding NaN values)
        valid_aucs = aucs[~np.isnan(aucs)]
        mean_auc = float(np.mean(valid_aucs)) if len(valid_aucs) > 0 else np.nan
        
        # Compute drift metrics (using first day as baseline)
        baseline_windows = len(data_dict["test_days"][0][1]) // eval_window if data_dict["test_days"] else 10
        baseline_acc = accs[:baseline_windows].mean() if len(accs) > baseline_windows else accs.mean()
        
        results[method_name]["summary"] = {
            "mean_accuracy": float(accs.mean()),
            "mean_f1": float(f1s.mean()),
            "mean_detection_rate": float(drs.mean()),
            "mean_precision": float(precs.mean()),
            "mean_auc_roc": mean_auc,
            "overall_auc_roc": overall_auc,
            "accuracy_std": float(accs.std()),
            "f1_std": float(f1s.std()),
            "dr_std": float(drs.std()),
            "baseline_accuracy": float(baseline_acc),
            "accuracy_drop": float(baseline_acc - accs.min()) if len(accs) > 0 else 0.0,
            "recovery_time": int(compute_recovery_time(accs, baseline_acc, baseline_windows)) * eval_window,
            "stability_index": float(accs.var()),
            "n_drift_detections": len(results[method_name]["drift_detections"]),
            "total_samples_evaluated": len(all_labels)
        }
        
        summary = results[method_name]["summary"]
        auc_str = f"{summary['overall_auc_roc']:.4f}" if not np.isnan(summary['overall_auc_roc']) else "N/A"
        n_drifts = summary['n_drift_detections']
        logger.info(f"{methods[method_name]['name']:25s}: "
                   f"Acc={summary['mean_accuracy']:.4f} ± {summary['accuracy_std']:.4f}, "
                   f"DR={summary['mean_detection_rate']:.4f}, "
                   f"AUC={auc_str}, "
                   f"Drifts={n_drifts}, "
                   f"Recovery={summary['recovery_time']} steps")
    
    # Log drift-aware method details
    for name in ["DDM_Retrain", "ADWIN_Retrain"]:
        if name in methods and hasattr(methods[name]["model"], "retrain_count"):
            model = methods[name]["model"]
            logger.info(f"  {name}: Retrains={model.retrain_count}, Drift points={model.drift_points[:5]}...")
    
    # Log Enhanced RL stats
    logger.info("\n--- ENHANCED RL + ADI COMPONENT STATS ---")
    
    if "Ours_RLSelect" in methods and hasattr(methods["Ours_RLSelect"]["model"], "get_drift_stats"):
        model = methods["Ours_RLSelect"]["model"]
        drift_stats = model.get_drift_stats()
        selection_stats = model.get_selection_stats()
        all_stats = model.get_all_stats()
        
        logger.info(f"  Enhanced RL Drift Detection:")
        logger.info(f"    Mean Drift: {drift_stats['mean_drift']:.4f}")
        logger.info(f"    Max Drift: {drift_stats['max_drift']:.4f}")
        logger.info(f"    Drift Detections: {drift_stats.get('drift_detection_count', 0)}")
        logger.info(f"    Majority Voting Used: {drift_stats.get('majority_voting_count', 0)}")
        logger.info(f"    Incremental Updates: {drift_stats.get('incremental_updates', 0)}")
        logger.info(f"    Drift Adaptations: {drift_stats['drift_triggered_adaptations']}")
        
        # ADI Stats
        logger.info(f"  ADI (Adversarial Drift Injection) Stats:")
        logger.info(f"    ADI Enabled: {drift_stats.get('adi_enabled', False)}")
        logger.info(f"    ADI Difficulty Adjustment: {drift_stats.get('adi_difficulty', 1.0):.4f}")
        logger.info(f"    ADI Attack Success Rate: {drift_stats.get('adi_attack_success_rate', 0.0):.4f}")
        
        logger.info(f"  Model Selection Distribution:")
        for model_name, count in selection_stats.items():
            logger.info(f"    {model_name}: {count}")
        
        # Store in results
        results["Ours_RLSelect"]["drift_stats"] = drift_stats
        results["Ours_RLSelect"]["selection_stats"] = selection_stats
        results["Ours_RLSelect"]["all_stats"] = all_stats
    
    return results


def evaluate_drift_scenario(
    config: DriftConfig,
    model_pool: Dict[str, nn.Module],
    q_network: nn.Module,
    X: np.ndarray,
    y: np.ndarray,
    device: torch.device
) -> Dict:
    """
    Evaluate drift adaptation with SYNTHETIC drift injection.
    Legacy function - use evaluate_temporal_drift for real drift.
    """
    np.random.seed(config.seed)
    
    # Inject drift
    injector = DriftInjector(seed=config.seed)
    X_drifted, y_drifted = injector.inject_drift(X, y, config)
    
    drift_point = config.drift_point
    eval_window = config.eval_window
    n_windows = len(X) // eval_window
    drift_window_idx = drift_point // eval_window
    
    results = {}
    
    # =========================================================================
    # Method 1: Static Best Model (no adaptation)
    # =========================================================================
    logger.info("  Evaluating Static Best Model...")
    best_model_name = list(model_pool.keys())[0]
    best_model = model_pool[best_model_name]
    
    static_accs = []
    for window_idx in range(n_windows):
        start_idx = window_idx * eval_window
        end_idx = min(start_idx + eval_window, len(X_drifted))
        
        X_window = torch.tensor(X_drifted[start_idx:end_idx], dtype=torch.float32, device=device)
        y_window = y_drifted[start_idx:end_idx]
        
        best_model.eval()
        with torch.no_grad():
            logits = best_model(X_window)
            preds = logits.argmax(dim=1).cpu().numpy()
        
        acc = accuracy_score(y_window, preds)
        static_accs.append(acc)
    
    static_accs = np.array(static_accs)
    baseline_acc = static_accs[:drift_window_idx].mean() if drift_window_idx > 0 else static_accs.mean()
    
    results["Static_Best"] = {
        "accuracies": static_accs.tolist(),
        "baseline_accuracy": float(baseline_acc),
        "metrics": asdict(compute_all_drift_metrics(static_accs, baseline_acc, drift_window_idx, config=config))
    }
    
    # =========================================================================
    # Method 2: DDM + Retrain
    # =========================================================================
    logger.info("  Evaluating DDM + Retrain...")
    ddm_model = copy.deepcopy(best_model)
    ddm_classifier = DriftAwareRetrainingClassifier(ddm_model, "DDM", device=device)
    
    ddm_accs = []
    for window_idx in range(n_windows):
        start_idx = window_idx * eval_window
        end_idx = min(start_idx + eval_window, len(X_drifted))
        
        X_window = X_drifted[start_idx:end_idx]
        y_window = y_drifted[start_idx:end_idx]
        
        preds = ddm_classifier.predict(X_window)
        acc = accuracy_score(y_window, preds)
        ddm_accs.append(acc)
        
        ddm_classifier.update_buffer(X_window, y_window)
        error = 1 - acc
        ddm_classifier.check_drift(error, start_idx)
    
    ddm_accs = np.array(ddm_accs)
    detection_point = ddm_classifier.drift_points[0] if ddm_classifier.drift_points else -1
    
    results["DDM_Retrain"] = {
        "accuracies": ddm_accs.tolist(),
        "baseline_accuracy": float(baseline_acc),
        "metrics": asdict(compute_all_drift_metrics(ddm_accs, baseline_acc, drift_window_idx, detection_point, config)),
        "drift_count": ddm_classifier.drift_count,
        "retrain_count": ddm_classifier.retrain_count
    }
    
    # =========================================================================
    # Method 3: ADWIN + Retrain
    # =========================================================================
    logger.info("  Evaluating ADWIN + Retrain...")
    adwin_model = copy.deepcopy(best_model)
    adwin_classifier = DriftAwareRetrainingClassifier(adwin_model, "ADWIN", device=device)
    
    adwin_accs = []
    for window_idx in range(n_windows):
        start_idx = window_idx * eval_window
        end_idx = min(start_idx + eval_window, len(X_drifted))
        
        X_window = X_drifted[start_idx:end_idx]
        y_window = y_drifted[start_idx:end_idx]
        
        preds = adwin_classifier.predict(X_window)
        acc = accuracy_score(y_window, preds)
        adwin_accs.append(acc)
        
        adwin_classifier.update_buffer(X_window, y_window)
        error = 1 - acc
        adwin_classifier.check_drift(error, start_idx)
    
    adwin_accs = np.array(adwin_accs)
    detection_point = adwin_classifier.drift_points[0] if adwin_classifier.drift_points else -1
    
    results["ADWIN_Retrain"] = {
        "accuracies": adwin_accs.tolist(),
        "baseline_accuracy": float(baseline_acc),
        "metrics": asdict(compute_all_drift_metrics(adwin_accs, baseline_acc, drift_window_idx, detection_point, config)),
        "drift_count": adwin_classifier.drift_count,
        "retrain_count": adwin_classifier.retrain_count
    }
    
    # =========================================================================
    # Method 4: Our RL Select Ensemble
    # =========================================================================
    logger.info("  Evaluating RL Select Ensemble (Ours)...")
    
    input_dim = X.shape[1]
    rl_ensemble = RLSelectEnsemble(
        models=model_pool, 
        device=device, 
        q_network=q_network,
        selection_temp=1.0,
        state_noise_std=0.1,
        seed=config.seed
    ).to(device)
    rl_ensemble.eval()
    rl_ensemble.enable_stochastic()
    
    rl_accs = []
    for window_idx in range(n_windows):
        start_idx = window_idx * eval_window
        end_idx = min(start_idx + eval_window, len(X_drifted))
        
        X_window = torch.tensor(X_drifted[start_idx:end_idx], dtype=torch.float32, device=device)
        y_window = y_drifted[start_idx:end_idx]
        
        with torch.no_grad():
            logits = rl_ensemble(X_window)
            preds = logits.argmax(dim=1).cpu().numpy()
        
        acc = accuracy_score(y_window, preds)
        rl_accs.append(acc)
    
    rl_accs = np.array(rl_accs)
    results["Ours_RLSelect"] = {
        "accuracies": rl_accs.tolist(),
        "baseline_accuracy": float(baseline_acc),
        "metrics": asdict(compute_all_drift_metrics(rl_accs, baseline_acc, drift_window_idx, config=config)),
        "model_selection_stats": rl_ensemble.get_selection_stats()
    }
    
    return results


def run_drift_evaluation():
    """Run full drift evaluation with real models - Q1 JOURNAL VERSION."""
    # Setup device with GPU info
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")
    
    # GPU performance tracking
    eval_start_time = time.time()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        print(f"⚡ GPU Accelerator Enabled: {torch.cuda.get_device_name(0)}")
        print(f"   Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        if USE_AMP:
            print(f"   Mixed Precision (AMP): Enabled")
    
    print("\n" + "=" * 80)
    print("🔬 DRIFT ADAPTATION EVALUATION - RL Co-Evolution IDS (Q1 Journal)")
    print("=" * 80)
    print("\nMục tiêu: Chứng minh hệ thống phục hồi nhanh sau concept drift")
    print("Methodology: Temporal split evaluation on CIC-IDS2017 (Monday→Friday)")
    print("-" * 80)
    
    all_results = {}
    
    # ==========================================================================
    # PART 1: TEMPORAL DRIFT EVALUATION (REAL DRIFT - PRIMARY)
    # ==========================================================================
    print("\n" + "=" * 80)
    print("📊 PART 1: TEMPORAL DRIFT EVALUATION (Real CIC-IDS2017 Drift)")
    print("=" * 80)
    print("Train: Tuesday (FTP/SSH-Patator), Test: Wed→Friday (different attack types)")
    print("NOTE: Monday only has BENIGN traffic, so we use Tuesday as baseline")
    
    # Load temporal data - Use Tuesday for training (has attacks)
    # This tests adaptation to NEW attack types (DoS, WebAttack, DDoS)
    data_dict = load_temporal_drift_data(
        DEFAULT_DATA_DIR, 
        DEFAULT_SCALER_PATH,
        train_days=["Tuesday"],  # Has FTP/SSH-Patator attacks
        test_days=["Wednesday", "Thursday", "Friday"],  # Different attacks = drift
        max_samples_per_day=8000,  # More samples for better evaluation
        seed=RANDOM_SEED
    )
    
    if len(data_dict["train"][0]) > 0:
        input_dim = data_dict["train"][0].shape[1]
        model_pool = load_model_pool(DEFAULT_MODEL_DIR, input_dim, device)
        
        state_dim = input_dim + 12 + 12 + 2
        q_net, loaded = load_q_network(DEFAULT_MODEL_DIR, DEFAULT_OUTPUT_DIR, state_dim, device)
        if not loaded:
            logger.warning("Q-Network not found, using random init for RL agent")
        
        # Run temporal evaluation
        temporal_results = evaluate_temporal_drift(
            model_pool=model_pool,
            q_network=q_net,
            data_dict=data_dict,
            device=device,
            eval_window=200
        )
        all_results["temporal_drift"] = temporal_results
        
        # Print temporal results table
        print("\n" + "=" * 80)
        print("📋 TEMPORAL DRIFT RESULTS - PER DAY BREAKDOWN")
        print("=" * 80)
        
        methods = list(temporal_results.keys())
        days = list(temporal_results[methods[0]]["per_day"].keys()) if methods else []
        
        # Header
        header = f"{'Method':<25}"
        for day in days:
            header += f" {day:<12}"
        header += f" {'Mean':<12}"
        print(header)
        print("-" * (25 + 13 * (len(days) + 1)))
        
        # Accuracy per day
        print("\n📊 Accuracy:")
        for method in methods:
            row = f"{method:<25}"
            accs = []
            for day in days:
                acc = temporal_results[method]["per_day"].get(day, {}).get("accuracy", 0)
                accs.append(acc)
                row += f" {acc:.4f}{'':>6}"
            row += f" {np.mean(accs):.4f}{'':>6}"
            print(row)
        
        # Detection Rate per day
        print("\n📊 Detection Rate:")
        for method in methods:
            row = f"{method:<25}"
            drs = []
            for day in days:
                dr = temporal_results[method]["per_day"].get(day, {}).get("detection_rate", 0)
                drs.append(dr)
                row += f" {dr:.4f}{'':>6}"
            row += f" {np.mean(drs):.4f}{'':>6}"
            print(row)
        
        # F1 per day
        print("\n📊 F1 Score:")
        for method in methods:
            row = f"{method:<25}"
            f1s = []
            for day in days:
                f1 = temporal_results[method]["per_day"].get(day, {}).get("f1", 0)
                f1s.append(f1)
                row += f" {f1:.4f}{'':>6}"
            row += f" {np.mean(f1s):.4f}{'':>6}"
            print(row)
        
        # Print Enhanced RL specific statistics
        print("\n" + "=" * 80)
        print("🚀 ENHANCED RL + ADI COMPONENT STATISTICS")
        print("=" * 80)
        
        if "Ours_RLSelect" in temporal_results:
            rl_method = temporal_results.get("Ours_RLSelect", {})
            
            # Print drift detection stats if available
            drift_detections = rl_method.get("drift_detections", [])
            drift_stats = rl_method.get("drift_stats", {})
            selection_stats = rl_method.get("selection_stats", {})
            
            print(f"\n📈 Enhanced RL Drift Detection Summary:")
            print(f"   Total drift points detected: {len(drift_detections)}")
            print(f"   Drift adaptations triggered: {drift_stats.get('drift_triggered_adaptations', 0)}")
            print(f"   Majority voting used: {drift_stats.get('majority_voting_count', 0)}")
            print(f"   Incremental updates: {drift_stats.get('incremental_updates', 0)}")
            print(f"   Mean drift magnitude: {drift_stats.get('mean_drift', 0):.4f}")
            print(f"   Max drift magnitude: {drift_stats.get('max_drift', 0):.4f}")
            
            # ADI Statistics
            print(f"\n🎯 ADI (Adversarial Drift Injection) Statistics:")
            print(f"   ADI Enabled: {drift_stats.get('adi_enabled', False)}")
            print(f"   ADI Difficulty Adjustment: {drift_stats.get('adi_difficulty', 1.0):.4f}")
            print(f"   ADI Attack Success Rate: {drift_stats.get('adi_attack_success_rate', 0.0):.2%}")
            if drift_stats.get('adi_attack_success_rate', 0.0) > 0.5:
                print(f"   → High attack success = model struggling, ADI increases difficulty")
            else:
                print(f"   → Low attack success = model robust, ADI decreases difficulty")
            
            if selection_stats:
                print(f"\n📊 Model Selection Distribution:")
                for model_name, count in selection_stats.items():
                    if isinstance(count, (int, float)) and count > 0:
                        print(f"   {model_name}: {int(count)}")
            
            # Print overall performance comparison
            print(f"\n📊 Overall Performance vs Baselines:")
            our_accs = [rl_method["per_day"][d].get("accuracy", 0) for d in days]
            our_mean = np.mean(our_accs)
            
            for method in methods:
                if method != "Ours_RLSelect":
                    other_accs = [temporal_results[method]["per_day"][d].get("accuracy", 0) for d in days]
                    other_mean = np.mean(other_accs)
                    improvement = (our_mean - other_mean) * 100
                    symbol = "↑" if improvement > 0 else "↓"
                    print(f"   vs {method:<20}: {symbol} {abs(improvement):.2f}% {'better' if improvement > 0 else 'worse'}")
    
    # ==========================================================================
    # PART 2: SYNTHETIC DRIFT EVALUATION (for completeness)
    # ==========================================================================
    print("\n" + "=" * 80)
    print("📊 PART 2: SYNTHETIC DRIFT EVALUATION (Controlled Experiments)")
    print("=" * 80)
    
    # Load data for synthetic drift
    X, y = load_full_data_for_drift(DEFAULT_DATA_DIR, DEFAULT_SCALER_PATH, num_samples=10000)
    if len(X) > 0:
        input_dim = X.shape[1]
        model_pool = load_model_pool(DEFAULT_MODEL_DIR, input_dim, device)
        
        state_dim = input_dim + 12 + 12 + 2
        q_net, loaded = load_q_network(DEFAULT_MODEL_DIR, DEFAULT_OUTPUT_DIR, state_dim, device)
        
        # Synthetic drift scenarios
        drift_configs = [
            DriftConfig(drift_type="sudden", drift_severity=1.0, drift_point=5000, eval_window=100),
            DriftConfig(drift_type="gradual", drift_severity=1.0, drift_point=5000, eval_window=100),
            DriftConfig(drift_type="recurring", drift_severity=1.0, drift_point=5000, eval_window=100),
            DriftConfig(drift_type="conditional", drift_severity=1.0, drift_point=5000, eval_window=100),
        ]
        
        for config in drift_configs:
            scenario_name = f"{config.drift_type}_severity{config.drift_severity}"
            print(f"\n--- Scenario: {scenario_name} ---")
            
            results = evaluate_drift_scenario(config, model_pool, q_net, X, y, device)
            all_results[f"synthetic_{scenario_name}"] = results
            
            # Print summary
            print(f"Recovery Time Comparison:")
            for method, data in results.items():
                recovery_time = data['metrics']['recovery_time']
                acc_drop = data['metrics']['accuracy_drop']
                final_acc = data['metrics']['final_accuracy']
                recovery_str = f"{recovery_time:6d}" if recovery_time > 0 else "  ∞   "
                print(f"  {method:20s}: Recovery={recovery_str} steps, Drop={acc_drop:.2%}, Final={final_acc:.2%}")
    
    # ==========================================================================
    # GENERATE VISUALIZATIONS
    # ==========================================================================
    print("\n" + "=" * 80)
    print("📊 GENERATING VISUALIZATIONS")
    print("=" * 80)
    
    if "temporal_drift" in all_results:
        plot_temporal_drift_results(all_results["temporal_drift"])
        # NEW: Plot drift τ and catastrophic forgetting
        plot_drift_tau_and_forgetting(all_results["temporal_drift"])
    
    # Save all results
    output_path = os.path.join(DEFAULT_OUTPUT_DIR, "drift_evaluation_results_q1.json")
    
    # Convert numpy arrays to lists for JSON serialization
    def convert_to_serializable(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(i) for i in obj]
        elif isinstance(obj, (np.int64, np.int32)):
            return int(obj)
        elif isinstance(obj, (np.float64, np.float32)):
            if np.isnan(obj):
                return None  # JSON doesn't support NaN
            return float(obj)
        elif isinstance(obj, float) and np.isnan(obj):
            return None
        return obj
    
    serializable_results = convert_to_serializable(all_results)
    with open(output_path, "w") as f:
        json.dump(serializable_results, f, indent=2)
    print(f"\n✅ Results saved to: {output_path}")
    
    # ==========================================================================
    # FINAL SUMMARY TABLE (LaTeX-ready)
    # ==========================================================================
    print("\n" + "=" * 80)
    print("📋 FINAL SUMMARY FOR Q1 JOURNAL")
    print("=" * 80)
    
    if "temporal_drift" in all_results:
        temporal = all_results["temporal_drift"]
        
        print("\n% LaTeX Table - Temporal Drift Results")
        print("\\begin{table*}[htbp]")
        print("\\centering")
        print("\\caption{Drift Adaptation Performance on CIC-IDS2017 (Temporal Split)}")
        print("\\label{tab:drift_temporal}")
        print("\\begin{tabular}{lccccccc}")
        print("\\toprule")
        print("\\textbf{Method} & \\textbf{Accuracy} & \\textbf{DR} & \\textbf{F1} & \\textbf{AUC-ROC} & \\textbf{Stability} & \\textbf{Recovery} & \\textbf{Drifts} \\\\")
        print("\\midrule")
        
        for method in temporal:
            summary = temporal[method].get("summary", {})
            acc = summary.get("mean_accuracy", 0)
            dr = summary.get("mean_detection_rate", 0)
            f1 = summary.get("mean_f1", 0)
            auc = summary.get("overall_auc_roc", np.nan)
            stability = summary.get("accuracy_std", 0)
            recovery = summary.get("recovery_time", -1)
            n_drifts = summary.get("n_drift_detections", 0)
            
            auc_str = f"{auc:.4f}" if not np.isnan(auc) else "-"
            recovery_str = f"{recovery:,}" if recovery > 0 else "$\\infty$"
            method_display = method.replace("_", "\\_")
            if method.startswith("Ours"):
                method_display = f"\\textbf{{{method_display}}}"
            
            print(f"{method_display} & {acc:.4f} & {dr:.4f} & {f1:.4f} & {auc_str} & {stability:.4f} & {recovery_str} & {n_drifts} \\\\")
        
        print("\\bottomrule")
        print("\\end{tabular}")
        print("\\end{table*}")
    
    # Performance summary
    eval_time = time.time() - eval_start_time
    print(f"\n{'='*80}")
    print(f"⏱️  EVALUATION PERFORMANCE SUMMARY")
    print(f"{'='*80}")
    print(f"Total evaluation time: {eval_time:.2f} seconds ({eval_time/60:.2f} minutes)")
    
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated() / 1e9
        print(f"Peak GPU memory used: {peak_memory:.2f} GB")
        print(f"GPU utilization: Optimized with AMP={USE_AMP}")
    
    return all_results


def plot_temporal_drift_results(results: Dict, save_path: str = None):
    """Generate comprehensive visualization for temporal drift evaluation."""
    if save_path is None:
        save_path = os.path.join(DEFAULT_OUTPUT_DIR, "temporal_drift_plot.png")
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    colors = {
        "Static_Best": "#E74C3C",
        "Ensemble_Voting": "#9B59B6",
        "DDM_Retrain": "#3498DB",
        "ADWIN_Retrain": "#2ECC71",
        "Sliding_Window": "#F39C12",
        "Incremental_SGD": "#1ABC9C",
        "Ours_RLSelect": "#E91E63",  # Our Enhanced RL (pink - stands out)
    }
    
    methods = list(results.keys())
    method_names = [m.replace("_", "\n") for m in methods]
    
    # Panel 1: Accuracy over time (all methods)
    ax1 = axes[0, 0]
    for method in methods:
        accs = results[method].get("accuracies_over_time", [])
        if accs:
            steps = np.arange(len(accs)) * 200  # eval_window
            ax1.plot(steps, accs, label=method.replace("_", " "), 
                    color=colors.get(method, 'gray'), linewidth=1.5, alpha=0.8)
    
    # Add day boundaries
    day_boundaries = results.get(methods[0], {}).get("day_boundaries", [])
    day_names = ["Tuesday", "Wednesday", "Thursday", "Friday"]
    for i, boundary in enumerate(day_boundaries):
        ax1.axvline(x=boundary, color='red', linestyle='--', alpha=0.5)
        if i < len(day_names):
            ax1.text(boundary + 100, 0.95, day_names[i], rotation=90, fontsize=8, alpha=0.7)
    
    ax1.set_xlabel('Sample Index', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title('Accuracy Over Time (Temporal Drift)', fontsize=14)
    ax1.legend(loc='lower left', fontsize=8, ncol=2)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0.3, 1.0])
    
    # Panel 2: Per-day performance comparison
    ax2 = axes[0, 1]
    days = list(results[methods[0]]["per_day"].keys()) if methods else []
    x = np.arange(len(days))
    width = 0.1
    
    for i, method in enumerate(methods[:7]):  # Max 7 methods for visibility
        accs = [results[method]["per_day"].get(day, {}).get("accuracy", 0) for day in days]
        offset = (i - 3) * width
        ax2.bar(x + offset, accs, width, label=method.replace("_", " "), 
               color=colors.get(method, 'gray'), alpha=0.8)
    
    ax2.set_xlabel('Day', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Per-Day Accuracy Comparison', fontsize=14)
    ax2.set_xticks(x)
    ax2.set_xticklabels(days, fontsize=10)
    ax2.legend(loc='lower left', fontsize=7, ncol=2)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim([0.5, 1.0])
    
    # Panel 3: AUC-ROC over time
    ax3 = axes[0, 2]
    for method in methods:
        aucs = results[method].get("auc_over_time", [])
        if aucs:
            steps = np.arange(len(aucs)) * 200
            valid_aucs = np.array(aucs)
            ax3.plot(steps, valid_aucs, label=method.replace("_", " "),
                    color=colors.get(method, 'gray'), linewidth=1.5, alpha=0.8)
    
    for i, boundary in enumerate(day_boundaries):
        ax3.axvline(x=boundary, color='red', linestyle='--', alpha=0.5)
    
    ax3.set_xlabel('Sample Index', fontsize=12)
    ax3.set_ylabel('AUC-ROC', fontsize=12)
    ax3.set_title('AUC-ROC Over Time', fontsize=14)
    ax3.legend(loc='lower left', fontsize=8, ncol=2)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0.3, 1.0])
    
    # Panel 4: Detection Rate comparison
    ax4 = axes[1, 0]
    mean_drs = []
    for method in methods:
        summary = results[method].get("summary", {})
        mean_drs.append(summary.get("mean_detection_rate", 0))
    
    bars = ax4.bar(range(len(methods)), mean_drs, 
                   color=[colors.get(m, 'gray') for m in methods], alpha=0.8)
    ax4.set_xticks(range(len(methods)))
    ax4.set_xticklabels(method_names, fontsize=8, rotation=0)
    ax4.set_ylabel('Mean Detection Rate', fontsize=12)
    ax4.set_title('Detection Rate Comparison', fontsize=14)
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_ylim([0.0, 1.0])
    
    for bar, val in zip(bars, mean_drs):
        ax4.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Panel 5: AUC-ROC comparison (bar chart)
    ax5 = axes[1, 1]
    mean_aucs = []
    for method in methods:
        summary = results[method].get("summary", {})
        auc_val = summary.get("overall_auc_roc", np.nan)
        mean_aucs.append(auc_val if not np.isnan(auc_val) else 0)
    
    bars = ax5.bar(range(len(methods)), mean_aucs,
                   color=[colors.get(m, 'gray') for m in methods], alpha=0.8)
    ax5.set_xticks(range(len(methods)))
    ax5.set_xticklabels(method_names, fontsize=8, rotation=0)
    ax5.set_ylabel('Overall AUC-ROC', fontsize=12)
    ax5.set_title('AUC-ROC Comparison', fontsize=14)
    ax5.grid(True, alpha=0.3, axis='y')
    ax5.set_ylim([0.0, 1.0])
    
    for bar, val in zip(bars, mean_aucs):
        if val > 0:
            ax5.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Panel 6: Stability comparison (accuracy std)
    ax6 = axes[1, 2]
    stability_vals = []
    for method in methods:
        summary = results[method].get("summary", {})
        stability_vals.append(summary.get("accuracy_std", 0))
    
    bars = ax6.bar(range(len(methods)), stability_vals,
                   color=[colors.get(m, 'gray') for m in methods], alpha=0.8)
    ax6.set_xticks(range(len(methods)))
    ax6.set_xticklabels(method_names, fontsize=8, rotation=0)
    ax6.set_ylabel('Accuracy Std Dev (lower is more stable)', fontsize=12)
    ax6.set_title('Stability Comparison', fontsize=14)
    ax6.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, stability_vals):
        ax6.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Temporal drift plot saved to: {save_path}")


def plot_drift_tau_and_forgetting(results: Dict, save_path: str = None):
    """
    Generate plots for:
    1. Drift Magnitude vs Adaptive Tau (τ) - shows tau increases when drift occurs
    2. Catastrophic Forgetting Metric - performance on old data after adaptation
    """
    if save_path is None:
        save_path = os.path.join(DEFAULT_OUTPUT_DIR, "drift_tau_forgetting_plot.png")
    
    # Get Enhanced RL data
    rl_data = results.get("Ours_RLSelect", {})
    tau_history = rl_data.get("tau_history", [])
    drift_history = rl_data.get("drift_magnitude_history", [])
    forgetting_scores = rl_data.get("forgetting_scores", [])
    day_boundaries = rl_data.get("day_boundaries", [])
    
    if not tau_history or not drift_history:
        print("⚠️ No tau/drift history available for plotting")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Panel 1: Drift Magnitude and Adaptive Tau over time (dual y-axis)
    ax1 = axes[0, 0]
    ax1_twin = ax1.twinx()
    
    steps = np.arange(len(drift_history))
    
    # Plot drift magnitude
    line1, = ax1.plot(steps, drift_history, 'b-', linewidth=1.5, alpha=0.7, label='Drift Magnitude')
    ax1.fill_between(steps, 0, drift_history, alpha=0.3, color='blue')
    ax1.axhline(y=0.20, color='orange', linestyle='--', alpha=0.7, label='Low Threshold (0.20)')
    ax1.axhline(y=0.30, color='red', linestyle='--', alpha=0.7, label='High Threshold (0.30)')
    
    # Plot adaptive tau
    line2, = ax1_twin.plot(steps, tau_history, 'r-', linewidth=2, alpha=0.8, label='Adaptive τ')
    
    # Add day boundaries
    day_names = ["Wed", "Thu", "Fri"]
    for i, boundary in enumerate(day_boundaries):
        # Convert boundary to step index (boundary is sample count, steps are per-forward)
        step_idx = boundary // 200 if boundary > 0 else 0  # Approximate
        if step_idx < len(steps):
            ax1.axvline(x=step_idx, color='green', linestyle=':', alpha=0.5)
            if i < len(day_names):
                ax1.text(step_idx + 2, ax1.get_ylim()[1] * 0.95, day_names[i], fontsize=9, alpha=0.7)
    
    ax1.set_xlabel('Forward Pass Index', fontsize=12)
    ax1.set_ylabel('Drift Magnitude', fontsize=12, color='blue')
    ax1_twin.set_ylabel('Adaptive τ (Temperature)', fontsize=12, color='red')
    ax1.set_title('Drift Magnitude vs Adaptive Temperature (τ)', fontsize=14)
    
    # Combined legend
    lines = [line1, line2]
    labels = ['Drift Magnitude', 'Adaptive τ']
    ax1.legend(lines, labels, loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Tau distribution when drift is high vs low
    ax2 = axes[0, 1]
    
    if len(drift_history) > 0:
        drift_arr = np.array(drift_history)
        tau_arr = np.array(tau_history[:len(drift_arr)])
        
        # Split by drift level
        low_drift_mask = drift_arr < 0.20
        med_drift_mask = (drift_arr >= 0.20) & (drift_arr < 0.30)
        high_drift_mask = drift_arr >= 0.30
        
        tau_low = tau_arr[low_drift_mask] if low_drift_mask.sum() > 0 else []
        tau_med = tau_arr[med_drift_mask] if med_drift_mask.sum() > 0 else []
        tau_high = tau_arr[high_drift_mask] if high_drift_mask.sum() > 0 else []
        
        # Box plot
        data_to_plot = [tau_low, tau_med, tau_high]
        labels_box = [f'Low Drift\n(<0.20)\nn={len(tau_low)}', 
                      f'Medium Drift\n(0.20-0.30)\nn={len(tau_med)}',
                      f'High Drift\n(≥0.30)\nn={len(tau_high)}']
        
        bp = ax2.boxplot([d for d in data_to_plot if len(d) > 0], 
                        labels=[l for l, d in zip(labels_box, data_to_plot) if len(d) > 0],
                        patch_artist=True)
        
        colors_box = ['#3498DB', '#F39C12', '#E74C3C']
        for patch, color in zip(bp['boxes'], colors_box[:len(bp['boxes'])]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax2.set_ylabel('Adaptive τ Value', fontsize=12)
        ax2.set_title('τ Distribution by Drift Level', fontsize=14)
        ax2.grid(True, alpha=0.3, axis='y')
    
    # Panel 3: Catastrophic Forgetting - Accuracy on old data over time
    ax3 = axes[1, 0]
    
    if forgetting_scores:
        sample_indices = [fs['sample_idx'] for fs in forgetting_scores]
        acc_on_old = [fs['acc_on_old_data'] for fs in forgetting_scores]
        n_updates = [fs['n_updates'] for fs in forgetting_scores]
        
        # Plot accuracy on old data
        ax3.plot(sample_indices, acc_on_old, 'o-', color='#E74C3C', linewidth=2, 
                markersize=8, label='Acc on Pre-Drift Data')
        
        # Add reference line for initial accuracy (first point or 1.0)
        if acc_on_old:
            initial_acc = acc_on_old[0] if acc_on_old else 1.0
            ax3.axhline(y=initial_acc, color='green', linestyle='--', alpha=0.7, 
                       label=f'Initial Acc ({initial_acc:.3f})')
        
        # Annotate with update count
        for i, (x, y, n) in enumerate(zip(sample_indices, acc_on_old, n_updates)):
            ax3.annotate(f'U{n}', (x, y), textcoords="offset points", 
                        xytext=(0, 10), ha='center', fontsize=8)
        
        ax3.set_xlabel('Sample Index', fontsize=12)
        ax3.set_ylabel('Accuracy on Pre-Drift Data', fontsize=12)
        ax3.set_title('Catastrophic Forgetting: Performance on Old Data After Updates', fontsize=14)
        ax3.legend(loc='lower left', fontsize=10)
        ax3.grid(True, alpha=0.3)
        ax3.set_ylim([0, 1.05])
    else:
        ax3.text(0.5, 0.5, 'No forgetting data available\n(No incremental updates occurred)', 
                ha='center', va='center', fontsize=12, transform=ax3.transAxes)
        ax3.set_title('Catastrophic Forgetting Metric', fontsize=14)
    
    # Panel 4: Forgetting Score Summary
    ax4 = axes[1, 1]
    
    if forgetting_scores and len(forgetting_scores) >= 2:
        acc_values = [fs['acc_on_old_data'] for fs in forgetting_scores]
        
        # Calculate forgetting metrics
        initial_acc = acc_values[0]
        final_acc = acc_values[-1]
        min_acc = min(acc_values)
        forgetting_amount = initial_acc - final_acc
        max_forgetting = initial_acc - min_acc
        
        metrics = {
            'Initial Acc\non Old Data': initial_acc,
            'Final Acc\non Old Data': final_acc,
            'Forgetting\nAmount': forgetting_amount,
            'Max\nForgetting': max_forgetting,
            'Retention\nRate': final_acc / initial_acc if initial_acc > 0 else 0
        }
        
        x_pos = range(len(metrics))
        colors_bar = ['#2ECC71', '#3498DB', '#E74C3C', '#E74C3C', '#9B59B6']
        bars = ax4.bar(x_pos, list(metrics.values()), color=colors_bar, alpha=0.8)
        ax4.set_xticks(x_pos)
        ax4.set_xticklabels(list(metrics.keys()), fontsize=9)
        ax4.set_ylabel('Value', fontsize=12)
        ax4.set_title('Catastrophic Forgetting Summary', fontsize=14)
        ax4.grid(True, alpha=0.3, axis='y')
        
        for bar, val in zip(bars, metrics.values()):
            ax4.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold')
    else:
        # Show placeholder with explanation
        ax4.text(0.5, 0.5, 'Catastrophic Forgetting Summary\n\nMetrics calculated after\nmultiple incremental updates', 
                ha='center', va='center', fontsize=12, transform=ax4.transAxes)
        ax4.set_title('Catastrophic Forgetting Summary', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Drift τ and Forgetting plot saved to: {save_path}")


def plot_drift_adaptation_results(results: Dict, save_path: str = None):
    """Generate drift adaptation visualization for synthetic drift."""
    if save_path is None:
        save_path = os.path.join(DEFAULT_OUTPUT_DIR, "drift_adaptation_plot.png")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Use first scenario for detailed plots
    scenario_name = list(results.keys())[0]
    scenario_results = results[scenario_name]
    
    colors = {
        "Static_Best": "#E74C3C",
        "DDM_Retrain": "#3498DB",
        "ADWIN_Retrain": "#2ECC71",
        "Ours_RLSelect": "#E91E63"
    }
    
    # Panel 1: Accuracy over time
    ax1 = axes[0, 0]
    for method, data in scenario_results.items():
        accs = data['accuracies']
        steps = np.arange(len(accs)) * 100  # eval_window = 100
        ax1.plot(steps, accs, label=method, color=colors.get(method, 'gray'), linewidth=2)
    
    ax1.axvline(x=5000, color='red', linestyle='--', alpha=0.7, label='Drift Point')
    ax1.set_xlabel('Sample Index', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title(f'Accuracy Over Time - {scenario_name}', fontsize=14)
    ax1.legend(loc='lower right', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0.4, 1.0])
    
    # Panel 2: Recovery time comparison (bar chart)
    ax2 = axes[0, 1]
    methods = list(scenario_results.keys())
    recovery_times = [scenario_results[m]['metrics']['recovery_time'] for m in methods]
    max_rt = max([r for r in recovery_times if r > 0], default=10000)
    recovery_times_viz = [r if r > 0 else max_rt * 1.2 for r in recovery_times]
    
    bars = ax2.bar(range(len(methods)), recovery_times_viz, color=[colors.get(m, 'gray') for m in methods])
    ax2.set_xticks(range(len(methods)))
    ax2.set_xticklabels(methods, rotation=15, ha='right', fontsize=10)
    ax2.set_ylabel('Recovery Time (samples)', fontsize=12)
    ax2.set_title('Recovery Time Comparison (lower is better)', fontsize=14)
    
    for bar, val, orig in zip(bars, recovery_times_viz, recovery_times):
        label = f'{orig:,}' if orig > 0 else '∞'
        ax2.annotate(label, xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Panel 3: Accuracy drop comparison
    ax3 = axes[1, 0]
    acc_drops = [scenario_results[m]['metrics']['accuracy_drop'] * 100 for m in methods]
    bars = ax3.bar(range(len(methods)), acc_drops, color=[colors.get(m, 'gray') for m in methods])
    ax3.set_xticks(range(len(methods)))
    ax3.set_xticklabels(methods, rotation=15, ha='right', fontsize=10)
    ax3.set_ylabel('Accuracy Drop (%)', fontsize=12)
    ax3.set_title('Maximum Accuracy Drop (lower is better)', fontsize=14)
    
    for bar, val in zip(bars, acc_drops):
        ax3.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Panel 4: Final accuracy comparison
    ax4 = axes[1, 1]
    final_accs = [scenario_results[m]['metrics']['final_accuracy'] * 100 for m in methods]
    bars = ax4.bar(range(len(methods)), final_accs, color=[colors.get(m, 'gray') for m in methods])
    ax4.set_xticks(range(len(methods)))
    ax4.set_xticklabels(methods, rotation=15, ha='right', fontsize=10)
    ax4.set_ylabel('Final Accuracy (%)', fontsize=12)
    ax4.set_title('Final Accuracy After Recovery (higher is better)', fontsize=14)
    ax4.set_ylim([50, 100])
    
    for bar, val in zip(bars, final_accs):
        ax4.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Plot saved to: {save_path}")


# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    print("=" * 80)
    print("🚀 RL CO-EVOLUTION IDS - FULL EVALUATION (Q1 Journal)")
    print("=" * 80)
    
    # Run basic clean data evaluation
    print("\n[1/2] Running Clean Data Evaluation...")
    run_evaluation()
    
    # Run drift adaptation evaluation (main focus for Q1 journal)
    print("\n[2/2] Running Drift Adaptation Evaluation...")
    run_drift_evaluation()


2025-12-05 03:20:54,299 | INFO | Device: cuda


✅ Mixed Precision (AMP) enabled for faster GPU inference
✅ CUDA Device: Tesla P100-PCIE-16GB
   Memory: 17.1 GB
🚀 RL CO-EVOLUTION IDS - FULL EVALUATION (Q1 Journal)

[1/2] Running Clean Data Evaluation...


2025-12-05 03:21:08,743 | INFO | Loaded 50 samples, 76 features
2025-12-05 03:21:08,924 | INFO | Loaded mlp from /kaggle/input/trained-model/pytorch/default/9/mlp_model.pth
2025-12-05 03:21:08,941 | INFO | Loaded cnn from /kaggle/input/trained-model/pytorch/default/9/cnn_model.pth
2025-12-05 03:21:08,956 | INFO | Loaded tcn from /kaggle/input/trained-model/pytorch/default/9/tcn_model.pth
2025-12-05 03:21:09,093 | INFO | Loaded bilstm from /kaggle/input/trained-model/pytorch/default/9/bilstm_model.pth
2025-12-05 03:21:09,094 | INFO | Input dim: 76, State dim: 102
2025-12-05 03:21:09,201 | INFO | Loaded defender from /kaggle/input/trained-model/pytorch/default/9/defender_latest.pth
2025-12-05 03:21:09,202 | INFO | 
2025-12-05 03:21:09,202 | INFO | MODEL POOL VERIFICATION (no mutations)
2025-12-05 03:21:09,203 | INFO | ============================================================
2025-12-05 03:21:09,340 | INFO |   mlp: Acc=0.9000, DR=0.9000, Pred[0]=5, Pred[1]=45
2025-12-05 03:21:09,783 | 


📊 RL SELECT EVALUATION (Clean Data)
Metric     RL(σ=0.0)    RL(σ=0.1)    RL(σ=0.5)    RL(σ=1.0)   
--------------------------------------------------------------
Acc        0.8800       0.8800       0.8800       0.8800      
DR         0.8800       0.8800       0.8800       0.8800      
F1         0.9362       0.9362       0.9362       0.9362      
ECE        0.0865       0.0865       0.0865       0.0865      

--------------------------------------------------------------------------------
🏆 Best noise level: σ=0.0 with DR=0.8800

[2/2] Running Drift Adaptation Evaluation...
⚡ GPU Accelerator Enabled: Tesla P100-PCIE-16GB
   Available Memory: 17.1 GB
   Mixed Precision (AMP): Enabled

🔬 DRIFT ADAPTATION EVALUATION - RL Co-Evolution IDS (Q1 Journal)

Mục tiêu: Chứng minh hệ thống phục hồi nhanh sau concept drift
Methodology: Temporal split evaluation on CIC-IDS2017 (Monday→Friday)
--------------------------------------------------------------------------------

📊 PART 1: TEMPORAL DRIF

2025-12-05 03:21:15,615 | INFO |   Tuesday: 8000 samples, Attack ratio=3.24%, Attacks=['FTP-Patator', 'SSH-Patator']
2025-12-05 03:21:24,867 | INFO |   Wednesday: 8000 samples, Attack ratio=35.20%, Attacks=['DoS slowloris', 'DoS GoldenEye', 'DoS Slowhttptest', 'DoS Hulk', 'Heartbleed']
2025-12-05 03:21:30,374 | INFO |   Thursday: 16000 samples, Attack ratio=0.53%, Attacks=['Web Attack � Brute Force', 'Infiltration', 'Web Attack � Sql Injection', 'Web Attack � XSS']
2025-12-05 03:21:38,487 | INFO |   Friday: 24000 samples, Attack ratio=40.93%, Attacks=['DDoS', 'Bot', 'PortScan']
2025-12-05 03:21:38,495 | INFO | Loaded mlp from /kaggle/input/trained-model/pytorch/default/9/mlp_model.pth
2025-12-05 03:21:38,503 | INFO | Loaded cnn from /kaggle/input/trained-model/pytorch/default/9/cnn_model.pth
2025-12-05 03:21:38,511 | INFO | Loaded tcn from /kaggle/input/trained-model/pytorch/default/9/tcn_model.pth
2025-12-05 03:21:38,535 | INFO | Loaded bilstm from /kaggle/input/trained-model/pytorch/

...

Recovery Time Comparison:
  Static_Best         : Recovery=  ∞    steps, Drop=16.12%, Final=86.60%
  DDM_Retrain         : Recovery=  ∞    steps, Drop=16.12%, Final=95.00%
  ADWIN_Retrain       : Recovery=  ∞    steps, Drop=16.12%, Final=86.60%
  Ours_RLSelect       : Recovery=  ∞    steps, Drop=64.12%, Final=62.10%

--- Scenario: recurring_severity1.0 ---


2025-12-05 03:31:51,959 | INFO |     → Retraining triggered! Buffer size: 1000, Drift count: 1
2025-12-05 03:31:52,002 | INFO |   Evaluating ADWIN + Retrain...
2025-12-05 03:31:52,110 | INFO |   Evaluating RL Select Ensemble (Ours)...
2025-12-05 03:31:52,335 | INFO |   Evaluating Static Best Model...
2025-12-05 03:31:52,413 | INFO |   Evaluating DDM + Retrain...
2025-12-05 03:31:52,484 | INFO |     → Retraining triggered! Buffer size: 1000, Drift count: 1
2025-12-05 03:31:52,525 | INFO |   Evaluating ADWIN + Retrain...


Recovery Time Comparison:
  Static_Best         : Recovery=  ∞    steps, Drop=47.12%, Final=90.90%
  DDM_Retrain         : Recovery=  ∞    steps, Drop=45.12%, Final=91.40%
  ADWIN_Retrain       : Recovery=  ∞    steps, Drop=47.12%, Final=90.90%
  Ours_RLSelect       : Recovery=  ∞    steps, Drop=58.12%, Final=86.70%

--- Scenario: conditional_severity1.0 ---


2025-12-05 03:31:52,635 | INFO |   Evaluating RL Select Ensemble (Ours)...


Recovery Time Comparison:
  Static_Best         : Recovery=   500 steps, Drop=20.12%, Final=86.40%
  DDM_Retrain         : Recovery=   500 steps, Drop=20.12%, Final=87.60%
  ADWIN_Retrain       : Recovery=   500 steps, Drop=20.12%, Final=86.40%
  Ours_RLSelect       : Recovery=   500 steps, Drop=22.12%, Final=86.40%

📊 GENERATING VISUALIZATIONS
📊 Temporal drift plot saved to: /kaggle/working/temporal_drift_plot.png
📊 Drift τ and Forgetting plot saved to: /kaggle/working/drift_tau_forgetting_plot.png

✅ Results saved to: /kaggle/working/drift_evaluation_results_q1.json

📋 FINAL SUMMARY FOR Q1 JOURNAL

% LaTeX Table - Temporal Drift Results
\begin{table*}[htbp]
\centering
\caption{Drift Adaptation Performance on CIC-IDS2017 (Temporal Split)}
\label{tab:drift_temporal}
\begin{tabular}{lccccccc}
\toprule
\textbf{Method} & \textbf{Accuracy} & \textbf{DR} & \textbf{F1} & \textbf{AUC-ROC} & \textbf{Stability} & \textbf{Recovery} & \textbf{Drifts} \\
\midrule
Static\_Best & 0.8893 & 0.6236 & 0